# 📈 AI Stock Market Analyst - Trading Advisor Agent

## 🎯 Project Overview

**Goal:** Build an intelligent trading advisor that provides actionable insights for stock market decisions.

**Key Features:**
- 📊 Conditional RAG: Automatically decides when to use documents vs LLM knowledge
- 🎯 Relevance Score: Smart decision-making based on similarity scores
- 📈 Quantitative Evaluation: Compare RAG vs LLM performance using metrics
- 💼 Practical Trading Guidance: Answer questions about market events and trends

---

## 🛠️ Required Packages

### 📦 Core Dependencies
- **langchain**: LangChain framework (chain composition)
- **langchain-community**: PDF loaders and integrations
- **langchain-openai**: OpenAI API integration
- **docarray**: Pure Python vector search (stable, no dependency conflicts)
- **pypdf**: PDF text extraction
- **docx2txt**: DOCX text extraction
- **tiktoken**: Token counting (for cost estimation)
- **openai**: Official OpenAI SDK
- **python-dotenv**: Environment variable management

In [ ]:
# Cell 1
# %pip install --upgrade pip
# %pip install -U langchain langchain-community langchain-openai
# %pip install -U docarray pypdf docx2txt tiktoken openai python-dotenv
# %pip install -U sentence-transformers  # 🔑 For HuggingFace embeddings (baconnier/Finance2_embedding_small_en-V1.5)

## 🔐 OpenAI API Key Configuration

**What This Does:**
- Loads your OpenAI API key from a `.env` file in the project directory
- The `.env` file should contain: `OPENAI_API_KEY=your-key-here`
- The `python-dotenv` package automatically reads `.env` files

In [ ]:
# Cell 3: Load OpenAI API Key from .env file
# This cell loads environment variables (especially the API key) before any API calls

import os
from dotenv import load_dotenv

# Load all variables from .env file into environment
# The .env file should be in the same directory as this notebook
load_dotenv()

# Check if API key was successfully loaded
if os.environ.get("OPENAI_API_KEY"):
    print("✅ OpenAI API key loaded successfully from .env file")
else:
    print("❌ Warning: OPENAI_API_KEY not found in .env file")
    print("   Please create a .env file with: OPENAI_API_KEY=your-key-here")

## ⚙️ Configuration (Hyperparameters)

These are the main hyperparameters you can adjust to customize the Trading Advisor system.

**🔬 Hyperparameters:** Configuration values that control system behavior (they are not learned from data).

---

### 📊 Hyperparameters Worth Testing

**🔴 HIGH IMPACT (Test these first!)**

1. **RELEVANCE_THRESHOLD (0.65)** – Most important!
   - Role: Decides when to use RAG vs Smart Fallback.
   - Test range: 0.50 - 0.80
   - Effect: Higher = stricter (more LLM), Lower = more lenient (more RAG).

2. **FALLBACK_THRESHOLD (0.50)** – Second most important!
   - Role: Decides when to use LLM vs Smart Fallback.
   - Test range: 0.40 - 0.60
   - Effect: Higher = wider fallback zone, Lower = more direct LLM.

3. **LLM_CONFIDENCE_THRESHOLD (5)** – Controls "No Answer" responses.
   - Role: Minimum confidence required to return an answer.
   - Test range: 3 - 7
   - Effect: Higher = stricter (more "No Answer"), Lower = more permissive.

4. **SIGMOID_MIDPOINT (0.5)** – 🔴 Critical for relevance-score amplification.
   - Role: Center point of the sigmoid transformation curve.
   - Test range: 0.50 - 0.60
   - Effect: Higher = more queries are pushed into the "high" relevance region, Lower = more queries pushed low.
   - Key idea: Strongly shapes the distribution of relevance scores.

5. **SIGMOID_STEEPNESS (10)** – 🔴 Critical for how sharply we separate queries.
   - Role: Controls how steep the sigmoid transition is.
   - Test range: 10 - 25
   - Effect: Higher = sharper separation (clearer boundaries), Lower = smoother transition.
   - Key idea: Larger values create a clearer gap between relevant and irrelevant queries.

**🟡 MEDIUM IMPACT**

6. **CHUNK_SIZE (800)** – Affects retrieval quality.
   - Role: Size of document chunks for embeddings.
   - Test range: 400 - 1200
   - Effect: Smaller = more precise, Larger = more context per chunk.

7. **TOP_K_DOCUMENTS (4)** – Number of documents to retrieve.
   - Test range: 2 - 8
   - Effect: More documents = more context but also more noise.

**🟢 LOW IMPACT (Fine-tuning)**

8. **LLM_TEMPERATURE (0.3)** – Controls answer creativity.
   - Test range: 0.0 - 0.7
   - Effect: Higher = more creative/varied, Lower = more focused/consistent.

9. **CHUNK_OVERLAP (100)** – Preserves context across chunks.
   - Test range: 50 - 200
   - Effect: Higher = better context continuity, but more redundancy.

---

### 🧪 Suggested Experiment Plan

**Phase 1: Threshold Testing (Most important!)**
- Test `RELEVANCE_THRESHOLD`: [0.55, 0.60, 0.65, 0.70, 0.75]
- Test `FALLBACK_THRESHOLD`: [0.40, 0.45, 0.50, 0.55]
- Test `SIGMOID_MIDPOINT`: [0.45, 0.50, 0.55, 0.60]
- Test `SIGMOID_STEEPNESS`: [8, 10, 15, 20]
- Observe how many questions go to RAG vs LLM vs Fallback.
- Inspect the relevance-score distribution (finance vs non-finance queries should separate clearly).

**Phase 2: Confidence Testing**
- Test `LLM_CONFIDENCE_THRESHOLD`: [3, 5, 7]
- See when the system returns "No Answer".

**Phase 3: Retrieval Quality**
- Test `CHUNK_SIZE`: [400, 800, 1200]
- Test `TOP_K_DOCUMENTS`: [2, 4, 6]

**Phase 4: Fine-tuning**
- Test `LLM_TEMPERATURE`: [0.0, 0.3, 0.5]
- Test `CHUNK_OVERLAP`: [50, 100, 200]

---

### 🔍 What to Observe When Testing

For each hyperparameter change, check:

1. **Mode Distribution** (see output of Cell 18).

2. **Score Patterns**
   - Relevance scores for each question.
   - RAG Score vs LLM Score (in Fallback mode).
   - Overall improvement percentage (Cell 22).

3. **Answer Quality** (Cell 24)
   - Are RAG answers better than before?
   - Is LLM being used appropriately?
   - Do the answers make sense?

4. **Evaluation Metrics** (Cell 22)
   - Overall score improvement.
   - Specificity, Relevance, and Factuality scores.

In [ ]:
# Cell 5
# ========================================
# 📌 USER CONFIGURATION (Hyperparameters)
# ========================================
#
# 🔬 HYPERPARAMETER TUNING STRATEGY:
# ====================================
# Step 1: Optimize RAG answer quality
#   - Tuning parameters: CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_DOCUMENTS, LLM_TEMPERATURE
#   - Goal: Maximize answer quality by letting RAG use documents effectively
#
# Step 2: Optimize system-level parameters  
#   - Tuning parameters: RELEVANCE_THRESHOLD, FALLBACK_THRESHOLD, OFF_TOPIC_THRESHOLD,
#                    SIGMOID_MIDPOINT, SIGMOID_STEEPNESS, LLM_CONFIDENCE_THRESHOLD
#   - Goal: Find the best routing for diverse questions (RAG vs LLM vs Fallback)
#
# 📋 Question set: TEST_QUERIES (used consistently across all steps)
# ========================================

# ========================================
# 📁 Common configuration (used in both steps)
# ========================================
DOCS_FOLDER = "../docs"  # Folder containing market analysis files (PDFs and DOCX)
LLM_MODEL = "gpt-4o-mini"  # LLM model to use

# ========================================
# 🔧 STEP 1: RAG answer-quality tuning parameters
# ========================================
# Goal: Maximize answer quality by letting RAG use documents effectively
# Parameters to tune:
# 1️⃣ Document Splitting Settings (🟡 Medium Impact)
CHUNK_SIZE = 800        # Size of each chunk (characters) | 800, Test: 400, 600, 800, 1000, 1200
                        # Smaller → more precise, larger → more context
CHUNK_OVERLAP = 100     # Overlap between chunks (characters) | 100, Test: 50, 100, 150, 200
                        # Higher → better context continuity, but more redundancy

# 2️⃣ Retrieval Settings (🟡 Medium Impact)
TOP_K_DOCUMENTS = 4     # Number of docs to retrieve | 4, Test: 2, 4, 6, 8
                        # More documents = more context, but also more noise

# 3️⃣ LLM Temperature (🟢 Low Impact)
LLM_TEMPERATURE = 0.2   # Lower = focused, Higher = creative | 0.2, Test: 0.0, 0.1, 0.2, 0.4, 0.7
                        # Higher → more creative/varied, lower → more focused/consistent

# ========================================
# ⚙️ STEP 2: System-level tuning parameters
# ========================================
# Goal: Find the best routing strategy for diverse questions (RAG vs LLM vs Fallback)
# Parameters to tune:
# 1️⃣ Sigmoid Transformation Settings (🔴 HIGH IMPACT!)
SIGMOID_MIDPOINT = 0.5        # 🔴 CRITICAL! | Center point of sigmoid | 0.5, Test: 0.40, 0.42, 0.45, 0.48, 0.50, 0.52, 0.55, 0.58, 0.60
                              # Scores above this → boosted high (0.80-0.99)
                              # Scores below this → pushed low (0.01-0.20)
SIGMOID_STEEPNESS = 12        # 🔴 CRITICAL! | How sharp the transition is | 12, Test: 5, 8, 10, 12, 15, 18, 20
                              # Higher = sharper separation between relevant/irrelevant
                              # Lower = smoother transition

# 2️⃣ Conditional RAG Thresholds (🔴 HIGH IMPACT!)
RELEVANCE_THRESHOLD = 0.62  # 🔴 MOST IMPORTANT! | RAG-only threshold | 0.62, Test: 0.55, 0.58, 0.60, 0.62, 0.65, 0.68, 0.70, 0.72, 0.75, 0.80
                            # Score >= this value: use RAG directly (high confidence)
FALLBACK_THRESHOLD = 0.5   # 🔴 2nd MOST IMPORTANT! | Smart Fallback zone | 0.5, Test: 0.40, 0.42, 0.45, 0.48, 0.50, 0.52, 0.55, 0.60
                            # For scores in this range: try both RAG and LLM, compare, and choose the better answer
                            # Score 0.50-0.65: Try BOTH, compare, use better answer
OFF_TOPIC_THRESHOLD = 0.15  # 🔴 3rd IMPORTANT! | Off-topic rejection threshold | 0.15, Test: 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30
                            # Score < this value: immediately reject (clearly off-topic)
                            # Score 0.20–0.50: use LLM with a domain check (stock-related or reject)

# 3️⃣ LLM Confidence Settings (🔴 HIGH IMPACT!)
LLM_CONFIDENCE_THRESHOLD = 5  # 🔴 "NO_ANSWER" if LLM confidence < this value | 5, Test: 3, 4, 5, 6, 7
                              # it triggers the NO_ANSWER mode
                              # Lower = more permissive (more answers), Higher = stricter (more "no answer")


# 6️⃣ Test Questions (for evaluation)
TEST_QUERIES = [
    # 1. 🟢 TIER 1: RAG Mode (relevance >= RELEVANCE_THRESHOLD = 0.65)
    # Should use documents directly (specific facts from docs)
    "Summarize the main points of the US stock market in October 2025",
    # "What were the main economic events and Federal Reserve decisions during Trump's second term?",
    # "How did the Federal Reserve stand change in 2025",

    # 2. 🟢/🔵 TIER 2: Smart Fallback (FALLBACK_THRESHOLD ≤ relevance < RELEVANCE_THRESHOLD, i.e., 0.50-0.65)
    # Should compare RAG vs LLM (broader analysis)
    # Expected: Green if RAG wins, Blue if LLM wins
    "List the major companies mentioned in the October 2025 market review with their performance",
    # "What were the key market trends and sector performance during the weeks of October 2025?",
    # "What were the Federal Reserve's policy changes or interest rate decisions mentioned in the October 2025 market reviews?"    

    # 3. 🟠 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check (general stock knowledge, not in docs)
    # Expected: LLM answers (Orange color) - Stock-related
    "How should I trade in highly volatile stock market that we can not predict the market based on 2025 market review?",
    
    # 4. 🟠/🔴 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check, but may have low confidence (prediction is uncertain)
    # Expected: Orange if confidence ≥ 6/10 (answers with caveats), Red if confidence < 6/10 (NO_ANSWER mode)
    "What will AAPL stock price be in 2026 based on the current market situation?",
    
    # 5. 🔴 TIER 3: LLM Domain Check (OFF_TOPIC_THRESHOLD ≤ relevance < FALLBACK_THRESHOLD, i.e., 0.20-0.50)
    # Should use LLM with domain check, LLM detects off-topic and rejects it
    # Expected: Red color - Domain-based rejection (not stock-related)
    "Who will be the Apple CEO in 2030?",
    
    # 6. 🔴 TIER 4: Off-Topic Auto-Reject (relevance < OFF_TOPIC_THRESHOLD = 0.20)
    # Should immediately reject without LLM call (clearly not about stock trading)
    # Expected: Red color - Auto-reject
    "Explain the concept of quantum entanglement and its applications in quantum computing."
]

# ========================================
# 🛡️ VALIDATION CHECKS
# ========================================

# Check: FALLBACK_THRESHOLD must be < RELEVANCE_THRESHOLD
if FALLBACK_THRESHOLD >= RELEVANCE_THRESHOLD:
    raise ValueError(
        f"\n❌ ERROR: Invalid threshold configuration!\n"
        f"   FALLBACK_THRESHOLD ({FALLBACK_THRESHOLD}) must be < RELEVANCE_THRESHOLD ({RELEVANCE_THRESHOLD})\n"
        f"\n"
        f"   Why? The 'uncertain zone' is defined as:\n"
        f"   [FALLBACK_THRESHOLD, RELEVANCE_THRESHOLD)\n"
        f"\n"
        f"   Current settings would create an impossible zone:\n"
        f"   [{FALLBACK_THRESHOLD}, {RELEVANCE_THRESHOLD}) = INVALID\n"
        f"\n"
        f"   💡 Suggested fix:\n"
        f"   - Keep RELEVANCE_THRESHOLD = {RELEVANCE_THRESHOLD}\n"
        f"   - Set FALLBACK_THRESHOLD to something like {RELEVANCE_THRESHOLD - 0.10:.2f} or {RELEVANCE_THRESHOLD - 0.15:.2f}\n"
    )

# Validate OFF_TOPIC_THRESHOLD
if OFF_TOPIC_THRESHOLD >= FALLBACK_THRESHOLD:
    raise ValueError(
        f"\n❌ ERROR: OFF_TOPIC_THRESHOLD ({OFF_TOPIC_THRESHOLD}) must be < FALLBACK_THRESHOLD ({FALLBACK_THRESHOLD})\n"
        f"   Thresholds must be: OFF_TOPIC < FALLBACK < RELEVANCE\n"
    )

# Validate sigmoid parameters
if not (0.0 <= SIGMOID_MIDPOINT <= 1.0):
    raise ValueError(f"❌ SIGMOID_MIDPOINT must be between 0.0 and 1.0, got {SIGMOID_MIDPOINT}")
if SIGMOID_STEEPNESS <= 0:
    raise ValueError(f"❌ SIGMOID_STEEPNESS must be positive, got {SIGMOID_STEEPNESS}")

# ========================================
# 📊 Configuration Summary
# ========================================
print("=" * 80)
print("✅ Configuration Complete!")
print("=" * 80)
print(f"\n📂 Documents Folder: {DOCS_FOLDER}")
print(f"🤖 LLM Model: {LLM_MODEL}")
print("\n" + "=" * 80)
print("🔧 STEP 1 - RAG answer-quality tuning parameters:")
print("=" * 80)
print(f"   📏 Chunk Size: {CHUNK_SIZE} (Overlap: {CHUNK_OVERLAP})")
print(f"   📊 Top K Documents: {TOP_K_DOCUMENTS}")
print(f"   🌡️ LLM Temperature: {LLM_TEMPERATURE}")
print("\n" + "=" * 80)
print("⚙️  STEP 2 - System-level tuning parameters:")
print("=" * 80)
print(f"   🎯 RAG Threshold: {RELEVANCE_THRESHOLD} 🔴")
print(f"   🔄 Fallback Threshold: {FALLBACK_THRESHOLD} 🔴")
print(f"   🚫 Off-Topic Threshold: {OFF_TOPIC_THRESHOLD} 🔴")
print(f"   🧠 LLM Confidence Threshold: {LLM_CONFIDENCE_THRESHOLD} 🔴")
print(f"   📈 Sigmoid Midpoint: {SIGMOID_MIDPOINT} 🔴")
print(f"   📈 Sigmoid Steepness: {SIGMOID_STEEPNESS} 🔴")
print("\n" + "=" * 80)
print(f"📋 Test Questions: {len(TEST_QUERIES)} (shared across all steps)")
print("\n" + "=" * 80)
print("✅ Validation passed: OFF_TOPIC < FALLBACK < RELEVANCE")
print("=" * 80)

## 📊 Logging, Monitoring & Error Handling Setup

**Why add this before hyperparameter tuning?**

When you experiment with different hyperparameters, you need to:
1. 📝 **Log** what happens (for debugging).
2. ⏱️ **Monitor** performance (response time, cost).
3. 🛡️ **Handle errors** gracefully (API failures, timeouts).

---

### 🎯 What We Will Track

**Performance Metrics:**
- ⏱️ Response time per query.
- 💰 Token usage and estimated cost.
- 📊 Mode distribution (RAG vs LLM vs Fallback).

**Error Handling:**
- ⚠️ OpenAI API errors (rate limits, timeouts).
- 📁 Document loading errors (missing files, corrupt PDFs).
- 🔄 Automatic retry for transient failures.

**Experiment Tracking:**
- 💾 Automatically save results to a CSV file.
- 📈 Compare different hyperparameter configurations.
- 🎯 Find optimal settings based on data.

---

In [ ]:
# Cell 7: Logging, Monitoring & Error Handling Setup
# This cell sets up logging, cost tracking, error handling, and experiment tracking

import logging
import time
import csv
import os
from datetime import datetime
from functools import wraps
import tiktoken

# ========================================
# 📝 LOGGING SETUP
# ========================================

# Create logs directory if it doesn't exist
os.makedirs("logs", exist_ok=True)

# Configure logging
# Note: remove existing handlers before reconfiguring so the cell can be run multiple times safely
# Remove all existing handlers so repeated cell runs behave reliably
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    handler.close()  # 핸들러 리소스 정리

# Then apply the base configuration
log_filename = f'logs/trading_advisor_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()  # Also print to console
    ],
    force=True  # Python 3.8+: force reconfiguration to avoid duplicate handlers
)

logger = logging.getLogger(__name__)
logger.info(f"📝 Logging initialized: {log_filename}")

# ========================================
# 💰 COST TRACKING
# ========================================

class CostTracker:
    """
    Track token usage and estimated API costs.
    
    OpenAI pricing (as of 2024):
    - GPT-4o-mini: $0.150 per 1M input tokens, $0.600 per 1M output tokens
    """
    
    # Pricing per 1M tokens (USD)
    PRICING = {
        "gpt-5": {"input": 1.25, "output": 10.00},
        "gpt-5-mini": {"input": 0.25, "output": 2.00},
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},        
    }
    
    def __init__(self, model_name="gpt-4o-mini"):
        """Initialize cost tracker for a specific model."""
        self.model_name = model_name
        self.encoding = tiktoken.encoding_for_model(model_name)
        self.total_input_tokens = 0
        self.total_output_tokens = 0
    
    def count_tokens(self, text):
        """Count tokens in a text string."""
        return len(self.encoding.encode(text))
    
    def add_tokens(self, input_text, output_text):
        """
        Add token counts for input and output.
        
        Args:
            input_text: Input prompt text
            output_text: Model response text
        """
        input_tokens = self.count_tokens(input_text)
        output_tokens = self.count_tokens(output_text)
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        
        return input_tokens, output_tokens
    
    def get_cost(self):
        """Calculate total cost in USD."""
        if self.model_name not in self.PRICING:
            return 0.0  # Unknown model
        
        pricing = self.PRICING[self.model_name]
        input_cost = (self.total_input_tokens / 1_000_000) * pricing["input"]
        output_cost = (self.total_output_tokens / 1_000_000) * pricing["output"]
        
        return input_cost + output_cost
    
    def get_summary(self):
        """Get summary of token usage and cost."""
        return {
            "input_tokens": self.total_input_tokens,
            "output_tokens": self.total_output_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens,
            "estimated_cost_usd": self.get_cost()
        }
    
    def reset(self):
        """Reset token counters."""
        self.total_input_tokens = 0
        self.total_output_tokens = 0

# Initialize global cost tracker
cost_tracker = CostTracker(model_name=LLM_MODEL)

# ========================================
# 🛡️ ERROR HANDLING UTILITIES
# ========================================

def retry_on_api_error(max_retries=2, delay=2):
    """
    Decorator to retry function on API errors.
    
    Args:
        max_retries: Maximum number of retry attempts
        delay: Delay in seconds between retries
    
    Usage:
        @retry_on_api_error(max_retries=2, delay=2)
        def api_call():
            # Your API call here
            pass
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    error_type = type(e).__name__
                    
                    # Log the error
                    if attempt < max_retries:
                        logger.warning(f"⚠️ {error_type} on attempt {attempt + 1}/{max_retries + 1}: {str(e)}")
                        logger.info(f"🔄 Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        logger.error(f"❌ Failed after {max_retries + 1} attempts: {str(e)}")
            
            # If all retries failed, raise the last exception
            raise last_exception
        
        return wrapper
    return decorator

def safe_file_load(file_path, loader_class):
    """
    Safely load a file with error handling.
    
    Args:
        file_path: Path to file
        loader_class: LangChain loader class (PyPDFLoader or Docx2txtLoader)
    
    Returns:
        List of loaded documents, or empty list if failed
    """
    try:
        loader = loader_class(file_path)
        documents = loader.load()
        logger.info(f"✅ Loaded: {os.path.basename(file_path)} ({len(documents)} pages/sections)")
        return documents
    except FileNotFoundError:
        logger.error(f"❌ File not found: {file_path}")
        return []
    except PermissionError:
        logger.error(f"❌ Permission denied: {file_path}")
        return []
    except Exception as e:
        logger.error(f"❌ Error loading {os.path.basename(file_path)}: {type(e).__name__} - {str(e)}")
        return []

# ========================================
# 📊 EXPERIMENT TRACKING
# ========================================

class ExperimentTracker:
    """Track and save hyperparameter experiment results."""
    
    def __init__(self, csv_filename="hyperparameter_experiments.csv"):
        """Initialize experiment tracker."""
        self.csv_filename = csv_filename
        self.current_experiment = {}
        
        # Create CSV file with headers if it doesn't exist
        if not os.path.exists(csv_filename):
            with open(csv_filename, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=[
                    'timestamp',  # 1
                    # RAG Quality (Step 1)
                    'chunk_size', 'chunk_overlap', 'top_k', 'temperature',  # 2-5
                    # System Architecture (Step 2)
                    'relevance_threshold', 'fallback_threshold', 'off_topic_threshold',  # 6-8
                    'llm_confidence_threshold', 'sigmoid_midpoint', 'sigmoid_steepness',  # 9-11
                    # Performance
                    'avg_response_time_sec', 'total_tokens', 'estimated_cost_usd',  # 12-14
                    # Mode Distribution
                    'mode_rag_count', 'mode_fallback_count', 'mode_llm_domain_check_count',  # 15-17
                    'mode_off_topic_count', 'mode_no_answer_count', 'mode_error_count',  # 18-20
                    # Improvement Metrics
                    'avg_improvement_pct', 'improvement_std', 'improvement_min', 'improvement_max',  # 21-24
                    # Score Metrics
                    'avg_relevance_score', 'avg_rag_score', 'avg_llm_score',  # 25-27
                    'avg_rag_specificity', 'avg_rag_relevance', 'avg_rag_factuality',  # 28-30
                    'avg_llm_specificity', 'avg_llm_relevance', 'avg_llm_factuality',  # 31-33
                    'notes'  # 34
                ])
                writer.writeheader()
    
    def start_experiment(self, config):
        """
        Start tracking a new experiment.
        
        Args:
            config: Dictionary with hyperparameter settings
        """
        self.current_experiment = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            **config,
            'response_times': [],
            'mode_counts': {
                'RAG': 0, 
                'FALLBACK': 0, 
                'LLM_DOMAIN_CHECK': 0, 
                'OFF_TOPIC': 0,
                'NO_ANSWER': 0,
                'ERROR': 0
            }
        }
        logger.info(f"🧪 Starting experiment with config: {config}")
    
    def log_query(self, response_time, mode):
        """Log a single query result."""
        self.current_experiment['response_times'].append(response_time)
        self.current_experiment['mode_counts'][mode] += 1
    
    def save_experiment(self, improvement_pct=None, 
                       improvement_std=None, improvement_min=None, improvement_max=None,
                       avg_relevance_score=None, avg_rag_score=None, avg_llm_score=None,
                       avg_rag_specificity=None, avg_rag_relevance=None, avg_rag_factuality=None,
                       avg_llm_specificity=None, avg_llm_relevance=None, avg_llm_factuality=None,
                       notes=""):
        """
        Save experiment results to CSV with detailed metrics.
        
        Args:
            improvement_pct: Overall RAG improvement percentage (from evaluation)
            improvement_std: Standard deviation of improvement across queries
            improvement_min: Minimum improvement across queries
            improvement_max: Maximum improvement across queries
            avg_relevance_score: Average relevance score (0-1)
            avg_rag_score: Average RAG answer quality (1-10)
            avg_llm_score: Average LLM answer quality (1-10)
            avg_rag_specificity: Average RAG specificity score (1-10)
            avg_rag_relevance: Average RAG relevance score (1-10)
            avg_rag_factuality: Average RAG factuality score (1-10)
            avg_llm_specificity: Average LLM specificity score (1-10)
            avg_llm_relevance: Average LLM relevance score (1-10)
            avg_llm_factuality: Average LLM factuality score (1-10)
            notes: Any additional notes about this experiment
        """
        cost_summary = cost_tracker.get_summary()
        
        # Calculate averages
        avg_response_time = sum(self.current_experiment['response_times']) / len(self.current_experiment['response_times']) if self.current_experiment['response_times'] else 0
        
        # Prepare row data (order matches the fieldnames)
        row = {
            'timestamp': self.current_experiment['timestamp'],
            # RAG Quality (Step 1)
            'chunk_size': self.current_experiment.get('chunk_size'),
            'chunk_overlap': self.current_experiment.get('chunk_overlap'),
            'top_k': self.current_experiment.get('top_k'),
            'temperature': self.current_experiment.get('temperature'),
            # System Architecture (Step 2)
            'relevance_threshold': self.current_experiment.get('relevance_threshold'),
            'fallback_threshold': self.current_experiment.get('fallback_threshold'),
            'off_topic_threshold': self.current_experiment.get('off_topic_threshold'),
            'llm_confidence_threshold': self.current_experiment.get('llm_confidence_threshold'),
            'sigmoid_midpoint': self.current_experiment.get('sigmoid_midpoint'),
            'sigmoid_steepness': self.current_experiment.get('sigmoid_steepness'),
            'avg_response_time_sec': f"{avg_response_time:.2f}",
            'total_tokens': cost_summary['total_tokens'],
            'estimated_cost_usd': f"${cost_summary['estimated_cost_usd']:.4f}",
            'mode_rag_count': self.current_experiment['mode_counts']['RAG'],
            'mode_fallback_count': self.current_experiment['mode_counts']['FALLBACK'],
            'mode_llm_domain_check_count': self.current_experiment['mode_counts']['LLM_DOMAIN_CHECK'],
            'mode_off_topic_count': self.current_experiment['mode_counts']['OFF_TOPIC'],
            'mode_no_answer_count': self.current_experiment['mode_counts']['NO_ANSWER'],
            'mode_error_count': self.current_experiment['mode_counts']['ERROR'],
            'avg_improvement_pct': f"{improvement_pct:.1f}%" if improvement_pct else "N/A",
            'improvement_std': f"{improvement_std:.2f}%" if improvement_std is not None else "N/A",
            'improvement_min': f"{improvement_min:.1f}%" if improvement_min is not None else "N/A",
            'improvement_max': f"{improvement_max:.1f}%" if improvement_max is not None else "N/A",
            'avg_relevance_score': f"{avg_relevance_score:.3f}" if avg_relevance_score is not None else "N/A",
            'avg_rag_score': f"{avg_rag_score:.2f}" if avg_rag_score is not None else "N/A",
            'avg_llm_score': f"{avg_llm_score:.2f}" if avg_llm_score is not None else "N/A",
            'avg_rag_specificity': f"{avg_rag_specificity:.2f}" if avg_rag_specificity is not None else "N/A",
            'avg_rag_relevance': f"{avg_rag_relevance:.2f}" if avg_rag_relevance is not None else "N/A",
            'avg_rag_factuality': f"{avg_rag_factuality:.2f}" if avg_rag_factuality is not None else "N/A",
            'avg_llm_specificity': f"{avg_llm_specificity:.2f}" if avg_llm_specificity is not None else "N/A",
            'avg_llm_relevance': f"{avg_llm_relevance:.2f}" if avg_llm_relevance is not None else "N/A",
            'avg_llm_factuality': f"{avg_llm_factuality:.2f}" if avg_llm_factuality is not None else "N/A",
            'notes': notes
        }
        
        # Append to CSV
        with open(self.csv_filename, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            writer.writerow(row)
        
        logger.info(f"💾 Experiment saved to {self.csv_filename}")
        logger.info(f"📊 Summary: Avg time={avg_response_time:.2f}s, Cost=${cost_summary['estimated_cost_usd']:.4f}, Improvement={improvement_pct:.1f}%")

# Initialize global experiment tracker
experiment_tracker = ExperimentTracker()

print("✅ Logging, Monitoring & Error Handling configured successfully!")
print(f"📝 Logs will be saved to: logs/")
print(f"📊 Experiment results will be saved to: hyperparameter_experiments.csv")
print(f"💰 Cost tracking enabled for model: {LLM_MODEL}")

## 📖 How to Use Logging, Monitoring & Error Handling

### 🎯 What Was Added

**✅ Automatic Logging:**
- All operations are logged to `logs/trading_advisor_[timestamp].log`.
- Logs include timestamps, error messages, and performance metrics.

**✅ Cost Tracking:**
- Token usage for each query is automatically counted.
- Estimated API cost is calculated in real time.
- A summary is printed after each experiment.

**✅ Error Handling:**
- API failures automatically retry (up to 2 times).
- File loading errors are caught and written to the logs.
- User-friendly error messages are returned.

**✅ Experiment Tracking:**
- All hyperparameter experiments are saved to `hyperparameter_experiments.csv`.
- Includes: response time, token usage, cost, mode distribution, improvement %, and more.
- Easy to compare different settings in Excel.

---

### 🔍 Understanding the CSV Output

The `hyperparameter_experiments.csv` file contains the following columns (in order):

| #   | Column                      | Description                                             | Type       | Impact       |
|-----|-----------------------------|---------------------------------------------------------|------------|-------------|
| 1   | `timestamp`                 | When the experiment was run                            | Metadata   | -           |
| **Hyperparameter: RAG Quality (Step 1)** |||||
| 2   | `chunk_size`                | Document chunk size                                    | 🟡 MEDIUM  | RAG Quality |
| 3   | `chunk_overlap`             | Overlap size between chunks                            | 🟡 MEDIUM  | RAG Quality |
| 4   | `top_k`                     | Number of documents retrieved                          | 🟡 MEDIUM  | RAG Quality |
| 5   | `temperature`               | LLM temperature                                        | 🟢 LOW     | RAG Quality |
| **Hyperparameter: System Architecture (Step 2)** |||||
| 6   | `relevance_threshold`       | Threshold for using RAG directly                       | 🔴 HIGH    | System      |
| 7   | `fallback_threshold`        | Start of Smart Fallback zone                           | 🔴 HIGH    | System      |
| 8   | `off_topic_threshold`       | Threshold for off-topic rejection                      | 🔴 HIGH    | System      |
| 9   | `llm_confidence_threshold`  | Confidence threshold for "No Answer"                   | 🔴 HIGH    | System      |
| 10  | `sigmoid_midpoint`         | Center point of sigmoid transformation                 | 🔴 CRITICAL| System      |
| 11  | `sigmoid_steepness`        | Sharpness of sigmoid transition                        | 🔴 CRITICAL| System      |
| **Performance Metrics** |||||
| 12  | `avg_response_time_sec`     | Average response time per query                        | Performance| -           |
| 13  | `total_tokens`              | Total tokens used (input + output)                     | Performance| -           |
| 14  | `estimated_cost_usd`        | Estimated API cost (USD)                               | Performance| -           |
| **Mode Distribution** |||||
| 15  | `mode_rag_count`            | Number of queries using RAG mode                       | Mode       | -           |
| 16  | `mode_fallback_count`      | Number of queries using Smart Fallback                 | Mode       | -           |
| 17  | `mode_llm_domain_check_count` | Number of queries using LLM Domain Check             | Mode       | -           |
| 18  | `mode_off_topic_count`      | Number of off-topic rejections                         | Mode       | -           |
| 19  | `mode_no_answer_count`      | Number of "No Answer" responses                        | Mode       | -           |
| 20  | `mode_error_count`          | Number of queries that resulted in errors              | Mode       | -           |
| **Improvement Metrics** |||||
| 21  | `avg_improvement_pct`       | RAG improvement over LLM-only (%)                      | Improvement| -           |
| 22  | `improvement_std`           | Standard deviation of improvement (stability)          | Improvement| -           |
| 23  | `improvement_min`           | Minimum improvement across queries                     | Improvement| -           |
| 24  | `improvement_max`           | Maximum improvement across queries                     | Improvement| -           |
| **Score Metrics** |||||
| 25  | `avg_relevance_score`       | Average relevance score (0-1)                          | Score      | -           |
| 26  | `avg_rag_score`            | Average RAG overall score (1-10)                       | Score      | -           |
| 27  | `avg_llm_score`            | Average LLM overall score (1-10)                       | Score      | -           |
| 28  | `avg_rag_specificity`      | Average RAG specificity (1-10)                         | Score      | -           |
| 29  | `avg_rag_relevance`        | Average RAG relevance (1-10)                           | Score      | -           |
| 30  | `avg_rag_factuality`       | Average RAG factuality (1-10)                          | Score      | -           |
| 31  | `avg_llm_specificity`      | Average LLM specificity (1-10)                         | Score      | -           |
| 32  | `avg_llm_relevance`        | Average LLM relevance (1-10)                           | Score      | -           |
| 33  | `avg_llm_factuality`       | Average LLM factuality (1-10)                          | Score      | -           |
| 34  | `notes`                     | Free-form experiment notes                             | Metadata   | -           |

## 📁 Document Loading Setup

**What This Section Does:**
This is a short configuration section that prepares for document loading. The actual loading happens in the next section.

**Environment:**
- Designed for local execution.
- Documents are read directly from the `docs/` folder.
- No manual file upload is required.

## 📄 Document Loading and Chunking

**What This Section Does:**
This is where all market analysis documents are actually loaded and processed.

**Step-by-step Process:**

1. **File Discovery** (`glob.glob`)
   - Scan the `docs/` folder for all PDF and DOCX files.
   - Build a list of file paths for each file type.

2. **Document Loading** (`PyPDFLoader`, `Docx2txtLoader`)
   - Convert PDF files to text (one page = one document).
   - Convert DOCX files to text (one section = one document).
   - Use the `safe_file_load()` wrapper so corrupted files do not crash the notebook.

3. **Text Chunking** (`RecursiveCharacterTextSplitter`)
   - Splits long documents into smaller chunks.
   - Each chunk is `CHUNK_SIZE` characters (default: 800).
   - Adds `CHUNK_OVERLAP` characters of overlap between chunks (default: 100).
   - Why overlap? To avoid losing context when sentences are split across chunk boundaries.

**Why We Chunk Documents:**
- **LLM Token Limits**: Models have a maximum input size (e.g., 128K tokens).
- **Better Retrieval**: Smaller chunks allow more accurate matching to a query.
- **Context Preservation**: Overlap helps ensure important information at boundaries is not lost.

**Example:**
- Original document length: 5000 characters.
- With `CHUNK_SIZE = 800` and `CHUNK_OVERLAP = 100`.
- Result: ~6–7 chunks with smooth transitions between them.

In [ ]:
# Cell 10: Load and Chunk Documents
# This cell loads all PDF and DOCX files from the docs folder and splits them into chunks

import os
import glob
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ========================================
# STEP 1: Find all PDF and DOCX files
# ========================================
logger.info(f"📂 Scanning folder: {DOCS_FOLDER}")

# Check if docs folder exists before proceeding
if not os.path.exists(DOCS_FOLDER):
    logger.error(f"❌ Docs folder not found: {DOCS_FOLDER}")
    raise FileNotFoundError(f"Documents folder '{DOCS_FOLDER}' does not exist")

# Use glob to find all PDF and DOCX files in the folder
# glob.glob() returns a list of file paths matching the pattern
pdf_files = glob.glob(os.path.join(DOCS_FOLDER, "*.pdf"))
docx_files = glob.glob(os.path.join(DOCS_FOLDER, "*.docx"))

print(f"📂 Found {len(pdf_files)} PDF files and {len(docx_files)} DOCX files in {DOCS_FOLDER}/")
logger.info(f"Found {len(pdf_files)} PDFs and {len(docx_files)} DOCX files")

# Check if any files found - warn if folder is empty
if len(pdf_files) == 0 and len(docx_files) == 0:
    logger.warning(f"⚠️ No PDF or DOCX files found in {DOCS_FOLDER}/")
    print(f"⚠️ WARNING: No documents found. Please add PDF or DOCX files to {DOCS_FOLDER}/")

# ========================================
# STEP 2: Load all documents into memory
# ========================================
all_documents = []  # This will store all loaded document pages/sections

# Load PDF files using PyPDFLoader (one page = one document)
# safe_file_load() wraps the loader with error handling (won't crash on corrupt files)
print(f"\n📄 Loading PDF files...")
for pdf_file in pdf_files:
    # safe_file_load() handles errors gracefully - returns [] on failure
    documents = safe_file_load(pdf_file, PyPDFLoader)
    all_documents.extend(documents)  # Add all pages from this PDF to our collection

# Load DOCX files using Docx2txtLoader (one section = one document)
print(f"\n📄 Loading DOCX files...")
for docx_file in docx_files:
    documents = safe_file_load(docx_file, Docx2txtLoader)
    all_documents.extend(documents)

# ========================================
# STEP 3: Validate that documents were loaded
# ========================================
if len(all_documents) == 0:
    logger.error("❌ No documents were successfully loaded")
    raise ValueError("Failed to load any documents. Please check file formats and permissions.")

print(f"\n✅ Total documents loaded: {len(all_documents)}")
logger.info(f"Successfully loaded {len(all_documents)} document sections")

# ========================================
# STEP 4: Split documents into chunks
# ========================================
# RecursiveCharacterTextSplitter splits text intelligently:
# - Tries to split at paragraph breaks first, then sentences, then words
# - Ensures chunks are approximately CHUNK_SIZE characters
# - Adds CHUNK_OVERLAP characters of overlap between chunks to preserve context
try:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,      # Target size: 800 characters per chunk
        chunk_overlap=CHUNK_OVERLAP  # Overlap: 100 characters between chunks
    )
    # Split all documents into chunks
    docs = text_splitter.split_documents(all_documents)
    print(f"🔹 Total chunks created: {len(docs)}")
    logger.info(f"Created {len(docs)} text chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
except Exception as e:
    logger.error(f"❌ Error splitting documents: {str(e)}")
    raise

## 🧠 Embedding and Vector Store Creation

**What This Section Does:**
Converts all document chunks into numerical vectors (embeddings) and builds a searchable index.

**Why We Need Embeddings:**
- Raw text cannot be compared directly by the computer.
- Embeddings convert text into numerical vectors that capture **semantic meaning**.
- Similar texts → similar vectors → we can find relevant documents by comparing vectors.

**Process:**

1. **Initialize the Embedding Model**
       - Uses HuggingFace model `baconnier/Finance2_embedding_small_en-V1.5`.
       - This is a **finance-specific** embedding model (not a general-purpose model).
       - Why? It better separates finance vs non-finance queries.

2. **Generate Vector Embeddings**
   - Each document chunk is converted into a vector (a high-dimensional numeric array).
       - These vectors represent the **semantic meaning** of the text.

3. **Build the Vector Store** (`DocArrayInMemorySearch`)
   - Stores all embeddings in memory for fast search.
   - Builds an index that enables similarity search.
   - No external database required – pure Python implementation.

**Next Steps:**
- When you ask a question, the query is also converted into an embedding.
- The system compares the query embedding against all document embeddings.
- It returns the most similar documents (highest cosine similarity scores).

**Key Advantages:**
- Goes beyond simple keyword matching (understands that "Apple" and "AAPL" are related).
- Captures semantic relationships (e.g., "Federal Reserve" vs "interest rates").

In [ ]:
# Cell 12: Create Embeddings and Vector Store
# This cell converts document chunks into embeddings and creates a searchable vector database

import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

# 🔑 Initialize finance-specific embeddings model
# Finance-specific embedding model initialization
# Model: baconnier/Finance2_embedding_small_en-V1.5
# - Trained specifically on financial texts for better domain separation
# - Because it is finance-specific, it separates finance vs non-finance queries more clearly
try:
    print("🔄 Loading finance-specific embedding model... (first time may take 1-2 minutes)")
    
    # Detect best device for your machine
    # MPS: Apple Silicon GPU (M1/M2/M3 Macs) - uses Metal framework for GPU acceleration
    # CUDA: NVIDIA GPU (Windows/Linux with NVIDIA cards) - very fast GPU acceleration
    # CPU: Universal fallback - slower but works everywhere (no GPU available)
    import torch
    if torch.backends.mps.is_available():
        device = 'mps'  # Apple Silicon GPU (MPS = Metal Performance Shaders)
        print("✅ Using Apple MPS (Metal Performance Shaders) - GPU acceleration")
    elif torch.cuda.is_available():
        device = 'cuda'  # NVIDIA GPU
        print("✅ Using CUDA (NVIDIA GPU) - GPU acceleration")
    else:
        device = 'cpu'  # CPU fallback (no GPU available)
        print("ℹ️ Using CPU (slower, but works on all machines - no GPU acceleration)")
    
        # 🏦 Initialize finance-specific embedding model
    # Model: baconnier/Finance2_embedding_small_en-V1.5
    # - Trained specifically on financial texts (Bloomberg, Reuters, financial reports)
    # - Provides better separation between finance vs non-finance queries
    # - More accurate semantic understanding for stock market domain
    # - normalize_embeddings=True ensures cosine similarity scores are in [0, 1] range
    embeddings = HuggingFaceEmbeddings(
        model_name="baconnier/Finance2_embedding_small_en-V1.5",  # Finance-specific embeddings
        model_kwargs={'device': device},  # Use detected device (GPU/CPU) for faster processing
        encode_kwargs={'normalize_embeddings': True}  # Normalize vectors for cosine similarity (0-1 range)
    )
    
    print(f"📊 Model loaded: Finance2_embedding_small_en-V1.5 (FINANCE-SPECIFIC)")
    print(f"🏦 This model is trained on financial texts for better domain separation")
    logger.info("✅ Finance-specific embeddings initialized")
    print("✅ Finance-specific embedding model loaded successfully")
except Exception as e:
    logger.error(f"❌ Error initializing finance-specific embeddings: {str(e)}")
    print(f"❌ Error: Could not load finance-specific embedding model. Please install: pip install sentence-transformers")
    raise

# ========================================
# STEP 1: Initialize finance-specific embedding model (completed above)
# ========================================
# The finance-specific embedding model is specialized for financial texts,
# which provides better separation between finance-related and non-finance queries
# (this is fully handled in the try-except block above).

# ========================================
# STEP 2: Convert all chunks to embeddings and create vector store
# ========================================
# This step converts each document chunk into a numerical vector (embedding)
# and creates a searchable index for fast similarity search
print(f"🔄 Embedding {len(docs)} document chunks... (this may take a while)")
logger.info(f"Starting embedding process for {len(docs)} chunks")

# Create vector store using DocArrayInMemorySearch
# This function is wrapped with @retry_on_api_error for automatic retry on failures
@retry_on_api_error(max_retries=2, delay=3)
def create_vectorstore():
    """
    Create vector store with retry on errors.
    
    What this does:
    - Converts each document chunk to an embedding vector
    - Stores all embeddings in memory for fast similarity search
    - Creates an index that allows finding similar documents by comparing vectors
    """
    start_time = time.time()
    
    # from_documents() creates a searchable vector index
    # - Takes all document chunks and embedding model
    # - Converts each chunk to a vector using the embeddings model
    # - Builds an index that enables fast similarity search
    vectorstore = DocArrayInMemorySearch.from_documents(
        documents=docs,      # All document chunks to embed
        embedding=embeddings,  # The finance-specific embedding model
    )
    
    elapsed_time = time.time() - start_time
    logger.info(f"✅ Vector store created in {elapsed_time:.1f} seconds")
    
    return vectorstore

try:
    vectorstore = create_vectorstore()
    print(f"✅ Vector store created successfully (DocArrayInMemorySearch)")
    print(f"📊 Total indexed chunks: {len(docs)}")
    logger.info(f"Vector store ready with {len(docs)} indexed chunks")
except Exception as e:
    logger.error(f"❌ Failed to create vector store: {str(e)}")
    print(f"❌ Error: Could not create vector store. Please check your API key and network connection.")
    raise

## 🎯 Conditional RAG System with Smart Fallback

**What This Is:**
This is the **core intelligence** of the Trading Advisor. It automatically decides the best way to answer each question.

**Problem It Solves:**
- Sometimes the answer is in the documents (use RAG).
- Sometimes the LLM’s general knowledge is better (use LLM-only).
- Sometimes it’s uncertain (compare both and pick the winner).

**How the Decision Process Works:**

1. **Compute the Relevance Score (0.0 - 1.0)**
   - Convert the query into an embedding.
   - Compare it against document embeddings.
   - Use a **sigmoid transformation** to amplify separation:
     - Finance-related queries → boosted into the 0.80–0.99 range.
     - Non-finance queries → pushed into the 0.01–0.20 range.
   - The average similarity after sigmoid = **Relevance Score**.

2. **Four-Tier Decision System:**

   **TIER 1: RAG Mode** (Relevance ≥ `RELEVANCE_THRESHOLD`)
   - Documents are highly relevant.
   - ✅ Use RAG directly (documents + LLM).
   - Result: 🟢 Green answer.

   **TIER 2: Smart Fallback** (`FALLBACK_THRESHOLD` ≤ Relevance < `RELEVANCE_THRESHOLD`)
   - Uncertain zone – documents might help, but not clearly.
   - ✅ Run both RAG and LLM-only.
   - ✅ Score both answers on quality (1–10 scale).
   - ✅ Choose the better answer.
   - Result: 🟢 Green (if RAG wins) or 🔵 Blue (if LLM wins).

   **TIER 3: LLM Domain Check** (`OFF_TOPIC_THRESHOLD` ≤ Relevance < `FALLBACK_THRESHOLD`)
   - Low relevance, but still possibly stock-related.
   - ✅ Ask the LLM using a domain-aware prompt.
   - ✅ LLM decides: stock-related (answer) or off-topic (reject).
   - ✅ Check LLM confidence – if too low, return "No Answer".
   - Result: 🟠 Orange (stock-related) or 🔴 Red (rejected / no answer).

   **TIER 4: Off-Topic Auto-Reject** (Relevance < `OFF_TOPIC_THRESHOLD`)
   - Clearly unrelated to finance/trading.
   - ✅ Immediately reject (no LLM call → saves cost).
   - Result: 🔴 Red (auto-rejected).

---

### 📊 Understanding the Three Different Scores

**1. Relevance Score (0.0 – 1.0)**
- **Purpose**: “Are the documents relevant to this query?”
- **How it’s computed:**
  1. Query → embedding vector.
  2. Compare with top-K document embeddings (cosine similarity).
  3. Compute the average similarity.
  4. Apply a sigmoid transformation (to amplify separation).
- **Usage**: Decides which mode to use.

**2. RAG Score (1–10)**
- **Purpose**: “How good is the RAG answer?”
- **How it’s computed:** The LLM evaluates the answer on three metrics and averages them:
  - **Specificity** (1–10): How detailed and concrete is the answer?
  - **Relevance** (1–10): Does it actually answer the question?
  - **Factuality** (1–10): Does it contain verifiable facts and data?
- **Usage**: Compare RAG vs LLM in Smart Fallback mode.

**3. LLM Score (1–10)**
- **Purpose**: “How good is the LLM-only answer?”
- **How it’s computed:** Same three metrics and averaging as for RAG Score.
- **Usage**: Compare RAG vs LLM in Smart Fallback mode.

**Key Distinction:**
- **Relevance Score** = quality of **document matching** (input quality).
- **RAG/LLM Scores** = quality of the **answers themselves** (output quality).
- These are **independent** – you can have high relevance but a poor answer, or low relevance but a strong LLM-only answer.

In [ ]:
# Cell 14: Conditional RAG Advisor Implementation
# This cell implements the ConditionalRAGAdvisor class with 4-tier decision system

import json
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from IPython.display import HTML, display
import numpy as np

class ConditionalRAGAdvisor:
    """
    Conditional RAG system that decides when to use documents vs LLM knowledge.
    
    Decision logic:
    - High relevance score (≥ threshold) → Use RAG (documents)
    - Low relevance score (< threshold) → Use LLM only
    - No confident answer → Return "Information not available"
    """
    
    def __init__(self, vectorstore, llm_model, relevance_threshold, fallback_threshold, off_topic_threshold,
                 top_k, llm_confidence_threshold, temperature, sigmoid_midpoint, sigmoid_steepness):
        """
        Initialize the Conditional RAG Advisor with Smart Fallback.
        
        Args:
            vectorstore: Vector database containing documents
            llm_model: Name of LLM model to use (e.g., "gpt-4o-mini")
            relevance_threshold: High confidence threshold for RAG-only (0.0-1.0)
            fallback_threshold: Low threshold for smart fallback (try both RAG and LLM)
            top_k: Number of documents to retrieve
            llm_confidence_threshold: Minimum LLM confidence (0-10) for valid answer
            temperature: LLM temperature (lower = more focused)
        """
        self.vectorstore = vectorstore
        self.relevance_threshold = relevance_threshold
        self.fallback_threshold = fallback_threshold
        self.off_topic_threshold = off_topic_threshold
        self.llm_confidence_threshold = llm_confidence_threshold
        self.top_k = top_k
        self.sigmoid_midpoint = sigmoid_midpoint
        self.sigmoid_steepness = sigmoid_steepness
        
        # Initialize LLM
        self.llm = ChatOpenAI(model=llm_model, temperature=temperature)
        
        # Create custom prompt for RAG chain
        # This forces the LLM to answer based on documents, not refuse due to date restrictions
        rag_prompt_template = """You are a financial analyst assistant. Answer the question based on the provided context documents.

INSTRUCTIONS:
1. Use the information from the context below to answer the question
2. Synthesize information from the context - you don't need exact quotes or lists
3. If the context discusses related topics, provide a helpful answer based on what's available
4. Only say "I don't have enough information" if the context is truly unrelated to the question
5. DO NOT refuse to answer based on date/time restrictions - the documents may contain information about any time period
6. Focus on being helpful - provide the best answer you can from the available context

Context: {context}

Question: {question}

Answer:"""
        
        RAG_PROMPT = PromptTemplate(
            template=rag_prompt_template,
            input_variables=["context", "question"]
        )
        
        # Create RAG chain with custom prompt
        self.rag_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(search_kwargs={"k": top_k}),
            chain_type_kwargs={"prompt": RAG_PROMPT}
        )
        
    def get_relevance_score(self, query):
        """
        Calculate the RELEVANCE SCORE for a query (0.0 - 1.0).
        
        This score answers: "How similar is the query to the documents in the database?"
        
        Step-by-Step Process:
        
        1. EMBEDDING CONVERSION:
           - Convert query text to embedding vector using finance-specific embedding model
           - Finance-specific 모델이므로 finance queries가 더 정확한 embeddings를 얻습니다
           - Query를 embedding vector로 변환
        
        2. SIMILARITY SEARCH:
           - Compare query embedding with all document embeddings
           - Use cosine similarity (measures angle between vectors)
           - Retrieve top-K most similar documents with their scores
        
        3. AVERAGE CALCULATION:
           - Calculate average similarity across top-K documents
           - This gives a stable, balanced relevance score (before transformation)
        
        4. SIGMOID TRANSFORMATION (KEY INNOVATION):
           - Formula: 1 / (1 + exp(-steepness * (avg_score - midpoint)))
           - Purpose: Amplify separation between relevant and irrelevant queries
           - How it works:
             * Scores above midpoint (0.5) → boosted toward 1.0 (0.80-0.99 range)
             * Scores below midpoint (0.5) → pushed toward 0.0 (0.01-0.20 range)
           - Result: Creates sharper boundaries for decision-making
        
        Decision Thresholds (after sigmoid):
        - Score ≥ RELEVANCE_THRESHOLD → Use RAG (documents highly relevant)
        - Score FALLBACK_THRESHOLD to RELEVANCE_THRESHOLD → Smart Fallback (compare both)
        - Score OFF_TOPIC_THRESHOLD to FALLBACK_THRESHOLD → LLM Domain Check (might be stock-related)
        - Score < OFF_TOPIC_THRESHOLD → Off-topic (auto-reject)
        
        Args:
            query: User's question text
        
        Returns:
            float: Relevance score (0.0-1.0) after sigmoid transformation
        """
        # STEP 1: Retrieve top-K documents with their similarity scores
        # similarity_search_with_score() compares query embedding with all document embeddings
        # Returns list of tuples: (document, similarity_score)
        docs_with_scores = self.vectorstore.similarity_search_with_score(query, k=self.top_k)
        
        # Edge case: No documents found
        if not docs_with_scores:
            return 0.0
        
        # STEP 2: Extract similarity scores from results
        # DocArrayInMemorySearch returns COSINE SIMILARITY scores (range: 0-1)
        # Higher score = more similar to the query
        scores = [score for _, score in docs_with_scores]
        
        # STEP 3: Calculate average similarity across top-K documents
        # Why average? More stable than just using the top-1 score
        # Accounts for multiple relevant documents
        avg_score = sum(scores) / len(scores)
        
        # STEP 4: Apply sigmoid transformation to amplify separation
        # This is the KEY INNOVATION that makes the system more accurate
        #
        # Before sigmoid: Finance queries might score 0.55, non-finance 0.45 (too close!)
        # After sigmoid:  Finance queries → 0.85, non-finance → 0.10 (clearly separated!)
        #
        # Sigmoid formula: 1 / (1 + exp(-steepness * (score - midpoint)))
        # - midpoint: Center point (default 0.5) - scores above get boosted, below get reduced
        # - steepness: How sharp the transition is (default 10) - higher = sharper separation
        import math
        sigmoid_score = 1 / (1 + math.exp(-self.sigmoid_steepness * (avg_score - self.sigmoid_midpoint)))
        
        return sigmoid_score
    
    def llm_with_domain_check(self, query):
        """
        Call LLM with domain-aware prompt to handle borderline relevance queries.
        
        This is used when relevance score is low but not zero (0.20-0.50).
        The LLM is instructed to ONLY answer stock trading questions and
        reject off-topic queries politely.
        
        Args:
            query: User's question
            
        Returns:
            LLM response (either an answer or rejection message)
        """
        domain_prompt = f"""You are a STOCK TRADING ADVISOR specializing in financial markets and trading.

IMPORTANT RULES:
1. You can ONLY answer questions about:
   - Stock markets, equities, trading strategies
   - Economic policies, Federal Reserve decisions, interest rates
   - Company earnings, financial analysis, valuations
   - Market trends, predictions, technical/fundamental analysis
   - Investment advice, portfolio management

2. If the question is about OTHER TOPICS (medicine, physics, sports, cooking, history, etc.):
   → Respond EXACTLY: "I'm a stock trading advisor and can only answer questions about financial markets and trading. Your question about [topic] is outside my expertise. Please ask about stock market or trading-related topics."

3. If you're unsure whether it's stock-related:
   → If it has ANY connection to finance/markets → answer it
   → If it's clearly unrelated → reject it politely

Question: {query}

Answer:"""
        
        try:
            response = self.llm.invoke(domain_prompt)
            answer = response.content if hasattr(response, 'content') else str(response)
            logger.info(f"🔍 Domain check LLM response: {answer[:100]}...")
            return answer
        except Exception as e:
            logger.error(f"❌ Error in domain check LLM: {str(e)}")
            return "I apologize, but I encountered an error. Please try again."
    
    def assess_llm_confidence(self, question, answer):
        """
        Ask the LLM to rate its own confidence in the answer.
        
        Uses LLM self-assessment to detect when the model is uncertain or hallucinating.
        
        Args:
            question: The original question
            answer: The LLM-generated answer
            
        Returns:
            Confidence score (0-10), or 0 if assessment fails
        """
        confidence_prompt = f"""Rate your confidence in the following answer on a scale of 0-10:

Question: {question}

Your Answer: {answer}

Instructions:
- If you provided specific, factual information you're confident about: 7-10
- If you provided general knowledge but aren't fully certain: 4-6  
- If you don't have enough information or are guessing: 0-3
- Be honest about your limitations

Respond with ONLY a single number from 0 to 10, nothing else.

Confidence score:"""
        
        try:
            response = self.llm.invoke(confidence_prompt).content.strip()
            # Extract first number found in response
            import re
            match = re.search(r'\d+', response)
            if match:
                confidence = int(match.group())
                # Clamp to 0-10 range
                confidence = max(0, min(10, confidence))
                return confidence
            else:
                return 0
        except Exception as e:
            print(f"Warning: Confidence assessment failed ({e}), assuming low confidence")
            return 0

    def score_answer(self, question, answer):
        """
        Calculate RAG SCORE or LLM SCORE (1-10) for an answer.
        
        This score measures: "How GOOD is this answer?" (NOT "Are documents relevant?")
        
        Evaluates answer quality on 3 dimensions:
        - Specificity: How detailed and specific?
        - Relevance: Does it directly answer the question?
        - Factuality: Does it contain verifiable facts and data?
        
        Final score = average of the 3 metrics
        
        IMPORTANT: This is completely independent of the Relevance Score!
        - Relevance Score (0-1) = "Do documents match query?" 
        - RAG/LLM Score (1-10) = "Is the answer high quality?"
        
        Returns:
            dict with specificity, relevance, and factuality scores (1-10)
        """
        evaluation_prompt = f"""You are an expert evaluator. Score the following answer on these three criteria (scale 1-10):

1. SPECIFICITY: How specific and detailed is the answer? (1=vague, 10=very specific with details)
2. RELEVANCE: How relevant is the answer to the question? (1=off-topic, 10=directly answers question)
3. FACTUALITY: Does the answer contain verifiable facts and data? (1=no facts/opinions only, 10=rich with facts and data)

Question: {question}

Answer: {answer}

Respond ONLY with a JSON object in this exact format (no other text):
{{"specificity": <score>, "relevance": <score>, "factuality": <score>}}"""
        
        try:
            response = self.llm.invoke(evaluation_prompt).content
            # Extract JSON from response (in case there's extra text)
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            json_str = response[start_idx:end_idx]
            scores = json.loads(json_str)
            return scores
        except:
            # Fallback if parsing fails
            return {"specificity": 5, "relevance": 5, "factuality": 5}

    def query(self, question):
        """
        Main query method - answers a question using Smart Fallback RAG logic.
        
        This is the CORE METHOD that implements the 4-tier decision system.
        
        Decision Flow:
        1. Calculate relevance score (with sigmoid transformation)
        2. Route to appropriate mode based on score:
           - TIER 1 (≥ RELEVANCE_THRESHOLD): RAG Mode
           - TIER 2 (FALLBACK_THRESHOLD to RELEVANCE_THRESHOLD): Smart Fallback
           - TIER 3 (OFF_TOPIC_THRESHOLD to FALLBACK_THRESHOLD): LLM Domain Check
           - TIER 4 (< OFF_TOPIC_THRESHOLD): Off-Topic Auto-Reject
        
        Args:
            question: User's question text
        
        Returns:
            dict: Contains answer, mode, scores, and metadata for display/analysis
        """
        # ========================================
        # STEP 1: Calculate relevance score
        # ========================================
        # This converts the question to an embedding, compares with documents,
        # and applies sigmoid transformation to get a relevance score (0.0-1.0)
        relevance_score = self.get_relevance_score(question)
        
        # Initialize result dictionary to store answer and metadata
        result = {
            "answer": None,              # The final answer text
            "mode": None,                 # Which mode was used (RAG, FALLBACK, etc.)
            "relevance_score": relevance_score,  # Relevance score (0.0-1.0)
            "llm_confidence": None,       # LLM's self-assessed confidence (0-10)
            "retrieved_docs": [],         # Documents retrieved from vector store
            "fallback_source": None,      # Which source won in FALLBACK mode (RAG or LLM)
            "rag_scores": None,          # RAG answer quality scores (in FALLBACK mode)
            "llm_scores": None           # LLM answer quality scores (in FALLBACK mode)
        }
        
        # ========================================
        # STEP 2: Route to appropriate mode based on relevance score
        # ========================================
        
        # TIER 1: RAG MODE (Relevance ≥ 0.65)
        # Documents are highly relevant - use RAG directly
        if relevance_score >= self.relevance_threshold:
            result["mode"] = "RAG"
            # Use RAG chain: retrieve documents + generate answer with LLM
            result["answer"] = self.rag_chain.invoke({"query": question})["result"]
            # Store retrieved documents for display
            result["retrieved_docs"] = self.vectorstore.similarity_search(question, k=self.top_k)
            
        # TIER 2: SMART FALLBACK (0.50 ≤ Relevance < 0.65)
        # Uncertain zone - try both RAG and LLM, compare answers, pick winner
        elif relevance_score >= self.fallback_threshold:
            result["mode"] = "FALLBACK"
            
            # Get RAG answer (documents + LLM)
            rag_answer = self.rag_chain.invoke({"query": question})["result"]
            result["retrieved_docs"] = self.vectorstore.similarity_search(question, k=self.top_k)
            
            # Get LLM-only answer (no documents, just LLM knowledge)
            llm_prompt = f"Answer the following question about stock market and trading:\n\nQuestion: {question}\n\nAnswer:"
            llm_answer = self.llm.invoke(llm_prompt).content
            
            # Assess LLM's confidence in its answer (0-10 scale)
            result["llm_confidence"] = self.assess_llm_confidence(question, llm_answer)
            
            # Decision Logic: If LLM is uncertain, prefer RAG (documents are more reliable)
            # Otherwise, score both answers and pick the better one
            if result["llm_confidence"] < self.llm_confidence_threshold:
                # LLM is uncertain - use RAG answer
                result["answer"] = rag_answer
                result["fallback_source"] = "RAG"
            else:
                # Both are confident - score both and compare
                # Score RAG answer on 3 metrics (specificity, relevance, factuality)
                rag_scores = self.score_answer(question, rag_answer)
                # Score LLM answer on same 3 metrics
                llm_scores = self.score_answer(question, llm_answer)
                
                # Store scores for display/analysis
                result["rag_scores"] = rag_scores
                result["llm_scores"] = llm_scores
                
                # Calculate overall scores (average of 3 metrics)
                rag_overall = np.mean(list(rag_scores.values()))
                llm_overall = np.mean(list(llm_scores.values()))
                
                # Pick the answer with higher overall score
                if rag_overall >= llm_overall:
                    result["answer"] = rag_answer
                    result["fallback_source"] = "RAG"
                else:
                    result["answer"] = llm_answer
                    result["fallback_source"] = "LLM"
        
        # TIER 3 & 4: Low relevance - determine if off-topic or stock-related
        else:
            # TIER 4: OFF-TOPIC AUTO-REJECT (Relevance < 0.15)
            # Clearly not about finance - reject immediately (saves API cost)
            if relevance_score < self.off_topic_threshold:
                # Tier 1: Very low relevance (< 0.20) → Clearly off-topic
                # Immediate rejection without LLM call (saves cost & latency)
                result["mode"] = "OFF_TOPIC"
                result["answer"] = (
                    "I'm a stock trading advisor and can only answer questions about financial markets and trading. "
                    "Your question appears to be outside my area of expertise. "
                    "Please ask about stock market, trading, economic policies, or financial analysis."
                )
                logger.info(f"🚫 Off-topic rejection: relevance={relevance_score:.3f} < {self.off_topic_threshold}")
                
            else:
                # Tier 2: Low but not zero relevance (0.20-0.50) → Borderline
                # Use LLM with domain-aware prompt to decide
                result["mode"] = "LLM_DOMAIN_CHECK"
                result["answer"] = self.llm_with_domain_check(question)
                
                # Assess LLM's confidence in its answer
                result["llm_confidence"] = self.assess_llm_confidence(question, result["answer"])
                
                # If LLM confidence is too low, switch to NO_ANSWER mode
                if result["llm_confidence"] < self.llm_confidence_threshold:
                    result["mode"] = "NO_ANSWER"
                    result["answer"] = f"Information not available. This question cannot be answered with confidence. (LLM confidence: {result['llm_confidence']}/10)"
                
                logger.info(f"🔍 Domain check mode: relevance={relevance_score:.3f}, confidence={result['llm_confidence']}")
        
        return result
    
# Initialize the Conditional RAG Advisor with Smart Fallback
advisor = ConditionalRAGAdvisor(
    vectorstore=vectorstore,
    llm_model=LLM_MODEL,
    relevance_threshold=RELEVANCE_THRESHOLD,
    fallback_threshold=FALLBACK_THRESHOLD,
    off_topic_threshold=OFF_TOPIC_THRESHOLD,
    top_k=TOP_K_DOCUMENTS,
    llm_confidence_threshold=LLM_CONFIDENCE_THRESHOLD,
    temperature=LLM_TEMPERATURE,
    sigmoid_midpoint=SIGMOID_MIDPOINT,
    sigmoid_steepness=SIGMOID_STEEPNESS
)

print("✅ Conditional RAG Advisor (Smart Fallback) initialized successfully!")
print(f"📊 Configuration: Model={LLM_MODEL}, RAG Threshold={RELEVANCE_THRESHOLD}, Fallback={FALLBACK_THRESHOLD}, Confidence={LLM_CONFIDENCE_THRESHOLD}, Top-K={TOP_K_DOCUMENTS}")

## 📊 Enhanced Query Function with Monitoring

**What This Section Does:**
Wraps the advisor’s `query` method with automatic monitoring, logging, and error handling.

**Why It’s Needed:**
- Track performance metrics (response time, cost, token usage).
- Handle errors gracefully (API failures, timeouts).
- Log experiment data for hyperparameter tuning.
- Systematically compare different configurations.

**What Gets Tracked:**
- ⏱️ **Response Time**: How long each query takes (seconds).
- 💰 **Token Usage & Cost**: Input/output tokens and estimated API cost.
- 📊 **Mode Distribution**: How many queries use each mode (RAG, Fallback, LLM, etc.).
- ⚠️ **Errors & Retries**: Automatic retries on transient failures (up to 2 attempts).

**How It Works:**
- The `monitored_query()` function wraps `advisor.query()`.
- All metrics are automatically logged to the experiment tracker.
- It returns the same structure as `advisor.query()` plus extra metadata.
- The `ask()` helper provides a convenient one-line interface for quick queries.

In [ ]:
# Cell 16: Enhanced Query Function with Monitoring
# This cell creates a wrapper function that adds monitoring, error handling, and cost tracking

def monitored_query(advisor, question, track_cost=True):
    """
    Enhanced query function with automatic monitoring and error handling.
    
    What This Function Does:
    1. Wraps advisor.query() with timing, error handling, and retry logic
    2. Tracks token usage and calculates estimated API cost
    3. Logs query results to experiment tracker for analysis
    4. Returns same format as advisor.query() + additional metadata
    
    Args:
        advisor: ConditionalRAGAdvisor instance (the main advisor object)
        question: User's question text
        track_cost: Whether to track token usage and cost (default: True)
    
    Returns:
        dict: Result dictionary with answer, metadata, response_time, token counts
    """
    start_time = time.time()  # Start timer for response time calculation
    result = None
    error_occurred = False
    
    try:
        logger.info(f"🔍 Processing query: {question[:50]}...")
        
        # Wrap advisor.query() with automatic retry on API errors
        # @retry_on_api_error automatically retries up to 2 times on failures
        @retry_on_api_error(max_retries=2, delay=2)
        def query_with_retry():
            return advisor.query(question)
        
        # Call the wrapped query method
        result = query_with_retry()
        
        # Calculate response time (elapsed time since start)
        response_time = time.time() - start_time
        result['response_time'] = response_time
        
        # Track token usage and cost if enabled
        if track_cost:
            # Estimate token usage based on input/output text length
            # cost_tracker uses tiktoken to count tokens accurately
            input_text = question
            output_text = result['answer']
            input_tokens, output_tokens = cost_tracker.add_tokens(input_text, output_text)
            
            # Store token counts in result for display/analysis
            result['input_tokens'] = input_tokens
            result['output_tokens'] = output_tokens
        
        # Log successful query completion with key metrics
        mode = result['mode']
        logger.info(f"✅ Query completed in {response_time:.2f}s | Mode: {mode} | Relevance: {result['relevance_score']:.3f}")
        
        # Track this query in experiment tracker for statistical analysis
        # This accumulates data for comparing different hyperparameter settings
        if hasattr(experiment_tracker, 'current_experiment') and experiment_tracker.current_experiment:
            experiment_tracker.log_query(response_time, mode)
        
        return result
        
    except Exception as e:
        # Error handling: Catch any exceptions and return error result
        error_occurred = True
        response_time = time.time() - start_time
        
        logger.error(f"❌ Query failed after {response_time:.2f}s: {type(e).__name__} - {str(e)}")
        
        # Return error result with user-friendly message
        return {
            'mode': 'ERROR',
            'answer': f"Sorry, an error occurred while processing your question: {str(e)}",
            'relevance_score': 0.0,
            'response_time': response_time,
            'error': str(e)
        }

# Create a convenience function that uses the global advisor
def ask(question):
    """
    Convenient shortcut function to ask a question with automatic monitoring.
    
    This is a simpler interface - just call ask(question) instead of 
    monitored_query(advisor, question).
    
    Usage:
        result = ask("What happened in the market?")
        print(result['answer'])
    
    Args:
        question: User's question text
    
    Returns:
        dict: Result dictionary with answer and metadata
    """
    return monitored_query(advisor, question)

print("✅ Monitored query function ready!")
print("💡 Use ask(question) for quick queries with automatic monitoring")

## 🧪 Testing the Conditional RAG System

We now test the advisor with sample questions to see how it chooses between RAG, Fallback, and LLM-only modes.

---
### 📊 Quick Reference: The Three Scores

```
┌─────────────────────────────────────────────────────────────────┐
│  RELEVANCE SCORE (0.0 - 1.0)                                    │
│  Question: "Do the documents match the query?"                  │
│  • 0.85 → Documents are highly relevant to the query ✓          │
│  • 0.54 → Documents somewhat match (uncertain zone)             │
│  • 0.30 → Documents are not relevant to the query ✗             │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  RAG SCORE (1-10)                                               │
│  Question: "How good is the RAG answer?"                        │
│  • Measures: Specificity + Relevance + Factuality               │
│  • 8.5 → High-quality answer with concrete facts from docs      │
│  • 6.0 → Decent answer but lacks detail                         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  LLM SCORE (1-10)                                               │
│  Question: "How good is the LLM-only answer?"                   │
│  • Measures: Same as RAG Score                                  │
│  • 8.0 → Good answer using the LLM’s pre-trained knowledge      │
│  • 4.0 → LLM does not have strong information on this topic     │
└─────────────────────────────────────────────────────────────────┘

💡 KEY INSIGHTS:
   1. Low Relevance + High LLM Score = Documents are irrelevant BUT the LLM still knows the answer.
   2. High Relevance + Low RAG Score = Documents match BUT the generated answer is low-quality.
```

In [ ]:
# Cell 18: Testing the Conditional RAG System
# This cell tests the advisor with sample questions and displays formatted results

# HTML display function for better formatting
def show_response(result, question):
    """
    Display the advisor's response in a formatted HTML box.
    Shows score comparison instead of LLM confidence for better transparency.
    """
    mode = result['mode']
    answer = result['answer']
    relevance_score = result['relevance_score']
    fallback_source = result.get('fallback_source')
    rag_scores = result.get('rag_scores')
    llm_scores = result.get('llm_scores')
    
    # Color coding based on mode
    if mode == "RAG":
        color = "#4caf50"  # Green
        icon = "🟢"
        mode_text = "RAG Mode (Using Documents)"
    elif mode == "FALLBACK":
        # In Fallback mode, color depends on the final answer source
        if fallback_source == "RAG":
            color = "#4caf50"  # Green
            icon = "🟢"
            mode_text = "RAG Answer (from Smart Fallback)"
        else: # LLM was chosen
            color = "#2196f3"  # Blue
            icon = "🔵"
            mode_text = "LLM Answer (from Smart Fallback)"
    elif mode == "LLM_DOMAIN_CHECK":
        # Check if LLM rejected the query as off-topic (domain rejection)
        # Be specific: only flag as off-topic if BOTH conditions met
        domain_rejection_keywords = [
            "stock trading advisor",
            "outside my expertise", 
            "outside my area"
        ]
        # Strong indicator of domain rejection
        has_domain_rejection = any(keyword.lower() in answer.lower() for keyword in domain_rejection_keywords)
        # Also mentions asking about stock/trading topics (redirecting user)
        has_redirection = "ask about stock" in answer.lower() or "trading-related topics" in answer.lower()
        
        is_off_topic_rejection = has_domain_rejection and has_redirection
        
        if is_off_topic_rejection:
            color = "#f44336"  # Red (rejected - not stock related)
            icon = "🔴"
            mode_text = "Off-Topic Question (Rejected by Domain Check)"
        else:
            color = "#ff9800"  # Orange (answered - stock related, even if with caveats)
            icon = "🟠"
            mode_text = "LLM Answer (Stock-Related, Low Document Relevance)"
    elif mode == "OFF_TOPIC":
        color = "#f44336"  # Red (auto-rejected - clearly off-topic)
        icon = "🔴"
        mode_text = "Off-Topic (Auto-Rejected)"
    elif mode == "NO_ANSWER":
        color = "#f44336"  # Red
        icon = "🔴"
        mode_text = "No Answer Available (Low Confidence)"
    elif mode == "ERROR":
        color = "#f44336"  # Red
        icon = "❌"
        mode_text = "System Error"
    else:
        color = "#9e9e9e"  # Grey
        icon = "⚪"
        mode_text = f"Unknown Mode: {mode}"
    
    # Build metadata line with relevance score
    metadata = f"{icon} <strong>{mode_text}</strong> | Relevance Score: {relevance_score:.3f}"
    
    # For FALLBACK mode, show score comparison instead of LLM confidence
    if mode == "FALLBACK" and rag_scores and llm_scores:
        # Calculate overall scores (average of specificity, relevance, factuality)
        rag_overall = sum(rag_scores.values()) / len(rag_scores)
        llm_overall = sum(llm_scores.values()) / len(llm_scores)
        
        # Show comparison
        if fallback_source == "RAG":
            metadata += f" | <span style='color:#4caf50;'><strong>RAG Score: {rag_overall:.1f}</strong></span> > LLM Score: {llm_overall:.1f}"
        else:
            metadata += f" | RAG Score: {rag_overall:.1f} < <span style='color:#2196f3;'><strong>LLM Score: {llm_overall:.1f}</strong></span>"
    
    html = f"""
    <div style="
        background-color:#f9f9f9;
        border-left: 6px solid {color};
        padding: 15px;
        margin: 15px 0;
        border-radius: 4px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;">
        <div style="color:#333; font-size:14px; margin-bottom:10px;">
            <strong>Question:</strong> {question}
        </div>
        <div style="color:#666; font-size:13px; margin-bottom:10px;">
            {metadata}
        </div>
        <div style="color:#111; font-size:14px; line-height:1.6; white-space: pre-wrap;">
            <strong>Answer:</strong><br>{answer}
        </div>
    </div>
    """
    display(HTML(html))

# Test with the configured test questions
print("🧪 Testing Conditional RAG Advisor with sample questions...\n")

# Start experiment tracking
experiment_tracker.start_experiment({
    # RAG Quality (Step 1)
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'top_k': TOP_K_DOCUMENTS,
    'temperature': LLM_TEMPERATURE,
    # System Architecture (Step 2)
    'relevance_threshold': RELEVANCE_THRESHOLD,
    'fallback_threshold': FALLBACK_THRESHOLD,
    'off_topic_threshold': OFF_TOPIC_THRESHOLD,
    'llm_confidence_threshold': LLM_CONFIDENCE_THRESHOLD,
    'sigmoid_midpoint': SIGMOID_MIDPOINT,
    'sigmoid_steepness': SIGMOID_STEEPNESS
})

# Reset cost tracker for this experiment
cost_tracker.reset()

# Store relevance scores for averaging later
relevance_scores = []

for i, question in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}/{len(TEST_QUERIES)}")
    print(f"{'='*80}")
    
    # Use monitored_query instead of advisor.query()
    result = monitored_query(advisor, question)
    show_response(result, question)
    
    # Store relevance score
    if 'relevance_score' in result:
        relevance_scores.append(result['relevance_score'])
    
    # Display performance metrics
    if 'response_time' in result:
        print(f"⏱️ Response time: {result['response_time']:.2f}s")
    if 'input_tokens' in result and 'output_tokens' in result:
        print(f"🎫 Tokens: {result['input_tokens']} input + {result['output_tokens']} output = {result['input_tokens'] + result['output_tokens']} total")

# Display experiment summary
print(f"\n{'='*80}")
print("📊 EXPERIMENT SUMMARY")
print(f"{'='*80}")
cost_summary = cost_tracker.get_summary()
print(f"💰 Total tokens: {cost_summary['total_tokens']:,}")
print(f"💵 Estimated cost: ${cost_summary['estimated_cost_usd']:.4f}")
print(f"📊 Mode distribution:")
for mode, count in experiment_tracker.current_experiment['mode_counts'].items():
    if count > 0:
        print(f"   - {mode}: {count}")

## 📊 Quantitative Evaluation: RAG vs LLM

**What This Section Does:**
Systematically compares answers generated with RAG against answers from an LLM-only baseline to measure performance.

**Why It Matters:**
- Provides quantitative evidence that RAG improves answer quality.
- Identifies when RAG helps and when it doesn’t.
- Supplies metrics to compare different hyperparameter settings.
- Clearly demonstrates your evaluation methodology (helpful for interviews and documentation).

**Evaluation Process:**

1. **For each test question:**
   - Generate an answer using RAG (documents + LLM).
   - Generate an answer using LLM-only (no documents).
   - Use LLM-as-a-judge to score both answers on three metrics.

2. **Evaluation Metrics** (each scored 1–10 by the LLM):
   - **Specificity** (1–10): How specific and detailed is the answer?
   - **Relevance** (1–10): Does it directly answer the question?
   - **Factuality** (1–10): Does it contain verifiable facts and data?
   - **Overall Score**: Average of the three metrics above.
   - **Word Count**: Length of the answer (for qualitative comparison only).

3. **Improvement Calculation:**
   - Per-question improvement: ((RAG score − LLM score) / LLM score) × 100%.
   - Average improvement across all questions.
   - Standard deviation, minimum, and maximum improvement (to measure consistency/stability).

**Key Insight:**
This evaluation shows whether RAG actually improves answers in practice, instead of just assuming it does.

In [ ]:
# Cell 20: Quantitative Evaluation - RAG vs LLM Comparison
# This cell implements the RAGEvaluator class for systematic comparison

import json
import pandas as pd

class RAGEvaluator:
    """
    Evaluates and compares RAG vs LLM-only responses.
    Uses LLM-as-a-judge for consistent scoring.
    """
    
    def __init__(self, llm, rag_chain):
        """
        Args:
            llm: ChatOpenAI instance for evaluation
            rag_chain: The advisor's RAG chain (with custom prompt)
        """
        self.llm = llm
        self.rag_chain = rag_chain
        
    def get_llm_only_answer(self, question):
        """
        Get answer using LLM only (no RAG).
        """
        prompt = f"""Answer the following question about stock market and trading using only your pre-trained knowledge:

Question: {question}

Answer:"""
        return self.llm.invoke(prompt).content
    
    def get_rag_answer(self, question):
        """
        Get answer using RAG (documents + LLM).
        Uses the advisor's RAG chain to ensure consistent behavior.
        """
        return self.rag_chain.invoke({"query": question})["result"]
    
    def score_answer(self, question, answer):
        """
        Score an answer on multiple metrics using LLM-as-a-judge.
        
        Returns:
            dict with specificity, relevance, and factuality scores (1-10)
        """
        evaluation_prompt = f"""You are an expert evaluator. Score the following answer on these three criteria (scale 1-10):

1. SPECIFICITY: How specific and detailed is the answer? (1=vague, 10=very specific with details)
2. RELEVANCE: How relevant is the answer to the question? (1=off-topic, 10=directly answers question)
3. FACTUALITY: Does the answer contain verifiable facts and data? (1=no facts/opinions only, 10=rich with facts and data)

Question: {question}

Answer: {answer}

Respond ONLY with a JSON object in this exact format (no other text):
{{"specificity": <score>, "relevance": <score>, "factuality": <score>}}"""
        
        try:
            response = self.llm.invoke(evaluation_prompt).content
            # Extract JSON from response (in case there's extra text)
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            json_str = response[start_idx:end_idx]
            scores = json.loads(json_str)
            return scores
        except:
            # Fallback if parsing fails
            return {"specificity": 5, "relevance": 5, "factuality": 5}
    
    def evaluate_comparison(self, question):
        """
        Compare RAG vs LLM-only for a single question.
        
        Returns:
            dict with both answers and their scores
        """
        # Get both answers
        llm_answer = self.get_llm_only_answer(question)
        rag_answer = self.get_rag_answer(question)
        
        # Score both answers
        llm_scores = self.score_answer(question, llm_answer)
        rag_scores = self.score_answer(question, rag_answer)
        
        # Calculate additional metrics
        llm_word_count = len(llm_answer.split())
        rag_word_count = len(rag_answer.split())
        
        return {
            "question": question,
            "llm_answer": llm_answer,
            "rag_answer": rag_answer,
            "llm_specificity": llm_scores["specificity"],
            "rag_specificity": rag_scores["specificity"],
            "llm_relevance": llm_scores["relevance"],
            "rag_relevance": rag_scores["relevance"],
            "llm_factuality": llm_scores["factuality"],
            "rag_factuality": rag_scores["factuality"],
            "llm_word_count": llm_word_count,
            "rag_word_count": rag_word_count,
        }

# Initialize evaluator with advisor's RAG chain
evaluator = RAGEvaluator(advisor.llm, advisor.rag_chain)

# Run evaluation on all test questions
print("📊 Running quantitative evaluation (this will take a few minutes)...\n")
evaluation_results = []

for i, question in enumerate(TEST_QUERIES, 1):
    print(f"Evaluating {i}/{len(TEST_QUERIES)}: {question[:50]}...")
    result = evaluator.evaluate_comparison(question)
    evaluation_results.append(result)
    print("  ✓ Complete")

print("\n✅ Evaluation complete!")

## 📈 Evaluation Results Visualization
Visualizes the comparison between RAG and LLM-only responses.

In [ ]:
# Cell 22: Evaluation Results Visualization
# This cell visualizes and analyzes the evaluation results comparing RAG vs LLM

# Convert results to DataFrame for easy analysis
df = pd.DataFrame(evaluation_results)

# Calculate average scores
avg_llm_specificity = df['llm_specificity'].mean()
avg_rag_specificity = df['rag_specificity'].mean()
avg_llm_relevance = df['llm_relevance'].mean()
avg_rag_relevance = df['rag_relevance'].mean()
avg_llm_factuality = df['llm_factuality'].mean()
avg_rag_factuality = df['rag_factuality'].mean()
avg_llm_words = df['llm_word_count'].mean()
avg_rag_words = df['rag_word_count'].mean()

# Calculate overall score (average of all three metrics)
df['llm_overall'] = (df['llm_specificity'] + df['llm_relevance'] + df['llm_factuality']) / 3
df['rag_overall'] = (df['rag_specificity'] + df['rag_relevance'] + df['rag_factuality']) / 3

avg_llm_overall = df['llm_overall'].mean()
avg_rag_overall = df['rag_overall'].mean()

# Display summary statistics
print("=" * 80)
print("📊 EVALUATION SUMMARY")
print("=" * 80)
print()
print(f"{'Metric':<25} {'LLM Only':<15} {'RAG':<15} {'Difference':<15}")
print("-" * 80)
print(f"{'Specificity (1-10)':<25} {avg_llm_specificity:<15.2f} {avg_rag_specificity:<15.2f} {avg_rag_specificity - avg_llm_specificity:+.2f}")
print(f"{'Relevance (1-10)':<25} {avg_llm_relevance:<15.2f} {avg_rag_relevance:<15.2f} {avg_rag_relevance - avg_llm_relevance:+.2f}")
print(f"{'Factuality (1-10)':<25} {avg_llm_factuality:<15.2f} {avg_rag_factuality:<15.2f} {avg_rag_factuality - avg_llm_factuality:+.2f}")
print(f"{'Overall Score (1-10)':<25} {avg_llm_overall:<15.2f} {avg_rag_overall:<15.2f} {avg_rag_overall - avg_llm_overall:+.2f}")
print(f"{'Average Word Count':<25} {avg_llm_words:<15.1f} {avg_rag_words:<15.1f} {avg_rag_words - avg_llm_words:+.1f}")
print()

# Calculate improvement percentage (OLD METHOD - for reference)
# This calculates: (avg of RAG scores - avg of LLM scores) / avg of LLM scores
improvement_aggregate = ((avg_rag_overall - avg_llm_overall) / avg_llm_overall) * 100

# We'll use the per-query method (calculated below) as the primary metric
# This is more accurate as it calculates improvement for each query individually

# Calculate improvement statistics per question (for stability analysis)
# THIS IS THE PRIMARY METHOD - more accurate and consistent
import numpy as np

# Calculate improvement percentage for each question
individual_improvements = []
for idx, row in df.iterrows():
    if row['llm_overall'] > 0:  # Avoid division by zero
        improvement_pct = ((row['rag_overall'] - row['llm_overall']) / row['llm_overall']) * 100
        individual_improvements.append(improvement_pct)

# Calculate statistics
if len(individual_improvements) > 0:
    improvement = np.mean(individual_improvements)  # PRIMARY METRIC (average of improvements)
    improvement_std = np.std(individual_improvements)
    improvement_min = min(individual_improvements)
    improvement_max = max(individual_improvements)
else:
    improvement = None
    improvement_std = None
    improvement_min = None
    improvement_max = None

# Display results
print(f"🎯 RAG Overall Improvement: {improvement:+.1f}%" if improvement is not None else "🎯 RAG Overall Improvement: N/A")
print(f"   (Calculated as: average of per-query improvements)")
print(f"   Alternative method (aggregate): {improvement_aggregate:+.1f}%")
print()

# Display detailed results table
print("=" * 80)
print("📋 DETAILED RESULTS BY QUESTION")
print("=" * 80)
print()

# Create a formatted table
result_table = df[['question', 'llm_overall', 'rag_overall']].copy()
result_table['improvement'] = result_table['rag_overall'] - result_table['llm_overall']
result_table['winner'] = result_table['improvement'].apply(lambda x: 'RAG ✓' if x > 0 else ('LLM ✓' if x < 0 else 'Tie'))

display(result_table)

# Display stability metrics (already calculated above)
if improvement is not None:
    print(f"\n{'='*80}")
    print("📊 IMPROVEMENT STABILITY ANALYSIS")
    print(f"{'='*80}")
    print(f"Average Improvement:  {improvement:+.1f}%")
    print(f"Standard Deviation:   {improvement_std:.2f}%")
    print(f"Minimum Improvement:  {improvement_min:+.1f}%")
    print(f"Maximum Improvement:  {improvement_max:+.1f}%")
    print(f"Range (Max - Min):    {improvement_max - improvement_min:.1f}%")
    print()
    
    # Stability interpretation
    if improvement_std < 10:
        stability = "🟢 Very Stable"
    elif improvement_std < 20:
        stability = "🟡 Moderately Stable"
    else:
        stability = "🔴 Unstable"
    
    print(f"Stability Rating: {stability}")
    print(f"💡 Lower std = more consistent across different queries")
    print()
    print(f"📝 Note: Two calculation methods:")
    print(f"   - Per-query method (used): {improvement:+.1f}% (average of individual improvements)")
    print(f"   - Aggregate method: {improvement_aggregate:+.1f}% (improvement of averages)")
    print(f"   - We use per-query method as it's more accurate for stability analysis")

# Save experiment results to CSV
print(f"\n{'='*80}")
print("💾 SAVING EXPERIMENT RESULTS")
print(f"{'='*80}")

# Calculate average relevance score from test queries
avg_relevance_score = sum(relevance_scores) / len(relevance_scores) if relevance_scores else None

experiment_tracker.save_experiment(
    improvement_pct=improvement,
    improvement_std=improvement_std,
    improvement_min=improvement_min,
    improvement_max=improvement_max,
    avg_relevance_score=avg_relevance_score,
    avg_rag_score=avg_rag_overall,
    avg_llm_score=avg_llm_overall,
    avg_rag_specificity=avg_rag_specificity,
    avg_rag_relevance=avg_rag_relevance,
    avg_rag_factuality=avg_rag_factuality,
    avg_llm_specificity=avg_llm_specificity,
    avg_llm_relevance=avg_llm_relevance,
    avg_llm_factuality=avg_llm_factuality,
    notes=f"Test run with {len(TEST_QUERIES)} questions"
)

print(f"\n✅ Experiment results saved to: hyperparameter_experiments.csv")
print(f"📊 Saved metrics:")
print(f"   - Avg Improvement: {improvement:+.1f}% (Std: {improvement_std:.2f}%, Min: {improvement_min:+.1f}%, Max: {improvement_max:+.1f}%)" if improvement_std is not None else f"   - Avg Improvement: {improvement:+.1f}%")
print(f"   - Avg Relevance Score: {avg_relevance_score:.3f}" if avg_relevance_score else "   - Avg Relevance Score: N/A")
print(f"   - Avg RAG Score: {avg_rag_overall:.2f}, Avg LLM Score: {avg_llm_overall:.2f}")
print(f"   - RAG (S/R/F): {avg_rag_specificity:.1f}/{avg_rag_relevance:.1f}/{avg_rag_factuality:.1f}")
print(f"   - LLM (S/R/F): {avg_llm_specificity:.1f}/{avg_llm_relevance:.1f}/{avg_llm_factuality:.1f}")
print(f"\n💡 Tip: Open the CSV file in Excel to compare different hyperparameter settings!")
print(f"📈 Pro Tip: Use 'improvement_std' column to find stable settings (lower std = more consistent)")


## 🔍 Side-by-Side Answer Comparison
See the actual answers from both RAG and LLM-only modes side by side.

In [ ]:
# Cell 24: Side-by-Side Answer Comparison
# This cell displays side-by-side comparison of RAG vs LLM answers for each question

def show_comparison(result):
    """
    Display side-by-side comparison of RAG vs LLM answers.
    Shows overall score (average of 3 metrics) plus individual scores.
    """
    question = result['question']
    llm_answer = result['llm_answer']
    rag_answer = result['rag_answer']
    
    # Calculate overall scores (average of the 3 metrics)
    llm_overall = (result['llm_specificity'] + result['llm_relevance'] + result['llm_factuality']) / 3
    rag_overall = (result['rag_specificity'] + result['rag_relevance'] + result['rag_factuality']) / 3
    
    # Format: Overall Score = X.X (Spec: X, Rel: X, Fact: X)
    llm_scores = f"Overall Score = {llm_overall:.1f} (Spec: {result['llm_specificity']}, Rel: {result['llm_relevance']}, Fact: {result['llm_factuality']})"
    rag_scores = f"Overall Score = {rag_overall:.1f} (Spec: {result['rag_specificity']}, Rel: {result['rag_relevance']}, Fact: {result['rag_factuality']})"
    
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif; margin: 20px 0;">
        <div style="background-color:#f5f5f5; padding:15px; border-radius:8px; margin-bottom:20px;">
            <h3 style="margin:0; color:#333;">Question:</h3>
            <p style="font-size:16px; color:#000; margin:10px 0 0 0;">{question}</p>
        </div>
        
        <div style="display:flex; gap:20px;">
            <!-- LLM Only -->
            <div style="flex:1; background-color:#fff; border:2px solid #2196f3; border-radius:8px; padding:15px;">
                <h4 style="margin:0 0 10px 0; color:#2196f3;">🔵 LLM Only</h4>
                <p style="font-size:12px; color:#666; margin:0 0 10px 0;">{llm_scores}</p>
                <div style="font-size:14px; line-height:1.6; color:#333; white-space:pre-wrap;">{llm_answer}</div>
            </div>
            
            <!-- RAG -->
            <div style="flex:1; background-color:#fff; border:2px solid #4caf50; border-radius:8px; padding:15px;">
                <h4 style="margin:0 0 10px 0; color:#4caf50;">🟢 RAG (Documents + LLM)</h4>
                <p style="font-size:12px; color:#666; margin:0 0 10px 0;">{rag_scores}</p>
                <div style="font-size:14px; line-height:1.6; color:#333; white-space:pre-wrap;">{rag_answer}</div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))

# Display comparison for each question
for i, result in enumerate(evaluation_results, 1):
    print(f"\n{'='*100}")
    print(f"Question {i}/{len(evaluation_results)}")
    print(f"{'='*100}\n")
    show_comparison(result)

## 💬 Interactive Q&A Interface
Now you can ask the Trading Advisor your own questions directly!

In [ ]:
# Cell 26: Interactive Q&A Interface
# This cell creates an interactive widget-based interface for asking questions

import ipywidgets as widgets
from IPython.display import clear_output

# Create interactive widgets
question_input = widgets.Textarea(
    value='',
    placeholder='Ask a question about the stock market (e.g., "What happened in the market last week?")',
    description='Your Question:',
    layout=widgets.Layout(width='80%', height='80px'),
    style={'description_width': '120px'}
)

submit_button = widgets.Button(
    description='Get Answer',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def on_submit_click(b):
    """
    Handle submit button click.
    """
    with output_area:
        clear_output()
        
        question = question_input.value.strip()
        
        if not question:
            print("⚠️ Please enter a question!")
            return
        
        print(f"🔍 Processing your question...\n")
        
        # Get answer from advisor with monitoring
        result = monitored_query(advisor, question)
        
        # Display the response
        show_response(result, question)
        
        # Display performance metrics
        if 'response_time' in result:
            print(f"\n⏱️ Response time: {result['response_time']:.2f}s")
        if 'input_tokens' in result and 'output_tokens' in result:
            total_cost = (result['input_tokens'] * 0.150 + result['output_tokens'] * 0.600) / 1_000_000
            print(f"🎫 Tokens: {result['input_tokens'] + result['output_tokens']} total")
            print(f"💵 Cost: ${total_cost:.4f}")
        
        # Show retrieved documents if RAG mode was used
        if result['mode'] == 'RAG' and result['retrieved_docs']:
            print("\n📄 Retrieved Documents:")
            print("-" * 80)
            for i, doc in enumerate(result['retrieved_docs'][:2], 1):  # Show top 2 docs
                preview = doc.page_content[:200].replace('\n', ' ')
                print(f"\n{i}. {preview}...")

# Attach button click handler
submit_button.on_click(on_submit_click)

# Display the interface
print("💬 Interactive Trading Advisor Interface")
print("=" * 80)
print("\nAsk questions about stock market, trading advice, or economic events.\n")

display(question_input)
display(submit_button)
display(output_area)

## 🎓 Summary and Key Learnings

### What We Built

✅ **Conditional RAG System with Smart Fallback**: 
   - A 4‑tier decision system that intelligently routes queries.
   - Uses a sigmoid transformation to amplify relevance scores.
   - Compares RAG vs LLM when uncertain and chooses the better answer.

✅ **Finance‑Specific Embeddings**: 
   - Uses the `baconnier/Finance2_embedding_small_en-V1.5` model (finance‑specific).
   - Provides better separation between finance and non‑finance queries.

✅ **Relevance Scoring with Sigmoid Amplification**: 
   - Computes cosine similarity between queries and documents.
   - Applies a sigmoid transformation to create sharper decision boundaries.
   - Finance queries are boosted into the 0.80–0.99 range, non‑finance into 0.01–0.20.

✅ **Quantitative Evaluation**: 
   - Uses LLM‑as‑a‑judge with three metrics (specificity, relevance, factuality).
   - Measures RAG improvement percentage and stability across queries.
   - Exports results to CSV for hyperparameter comparison.

✅ **Production‑Ready Features**: 
   - Automatic logging and error handling.
   - Cost tracking and token usage monitoring.
   - Experiment tracking for hyperparameter tuning.
   - Interactive Q&A interface.

---

### Main Components and Their Roles

**0. Project Overview and Setup** (`Cell 0`):
   - **Cell 0** (Markdown): Project overview, goals, key features, and required packages.

**1. Package Installation** (`Cell 1`):
   - **Cell 1** (Code): **Implementation** – installs core Python packages (langchain, docarray, pypdf, sentence‑transformers, etc.; commented so you can run manually when needed).

**2. API Key Setup** (`Cells 2–3`):
   - **Cell 2** (Markdown): Explains loading the API key from a `.env` file.
   - **Cell 3** (Code): **Implementation** – loads the OpenAI API key using `python-dotenv` and validates that it exists.

**3. Hyperparameter Configuration** (`Cells 4–5`):
   - **Cell 4** (Markdown): Describes all hyperparameters, their impact level (HIGH/MEDIUM/LOW), test ranges, experiment plan, and quick reference table.
   - **Cell 5** (Code): **Implementation** – defines all hyperparameters (thresholds, chunk size, LLM settings, sigmoid parameters), validates relationships between thresholds, defines test queries, and prints a configuration summary.

**4. Logging & Monitoring Setup** (`Cells 6–7`):
   - **Cell 6** (Markdown): Explains why logging/monitoring are needed and which metrics are tracked.
   - **Cell 7** (Code): **Implementation** – configures logging to file and console, creates a `CostTracker` class for token/cost tracking, defines a retry decorator, and implements an `ExperimentTracker` class to export results to CSV.

**5. Logging Usage Guide** (`Cell 8`):
   - **Cell 8** (Markdown): Explains how to use logging, interpret the CSV output, and understand stability metrics.

**6. Document Loading** (`Cells 9–10`):
   - **Cell 9** (Markdown): Describes the document loading setup.
   - **Cell 10** (Code): **Implementation** – scans the `docs/` folder, loads PDF/DOCX files using `PyPDFLoader`/`Docx2txtLoader`, and splits documents into overlapping chunks with `RecursiveCharacterTextSplitter`.

**7. Embedding & Vector Store** (`Cells 11–12`):
   - **Cell 11** (Markdown): Explains embedding creation and the vector store.
   - **Cell 12** (Code): **Implementation** – initializes the finance‑specific embedding model, detects device (MPS/CUDA/CPU), converts chunks to embeddings, and creates a `DocArrayInMemorySearch` vector store for fast similarity search.

**8. `ConditionalRAGAdvisor`** (`Cells 13–14`):
   - **Cell 13** (Markdown): Describes the 4‑tier decision system and the three different scores.
   - **Cell 14** (Code): **Implementation** – defines the `ConditionalRAGAdvisor` class:
     - `get_relevance_score()`: Computes relevance with a sigmoid transformation.
     - `llm_with_domain_check()`: Checks if the query is stock‑related.
     - `assess_llm_confidence()`: Has the LLM self‑assess its confidence.
     - `score_answer()`: Evaluates answer quality on specificity, relevance, and factuality.
     - `query()`: Core method implementing the 4‑tier routing logic (RAG/Fallback/LLM Domain Check/Off‑Topic).

**9. Monitoring & Query Functions** (`Cells 15–16`):
   - **Cell 15** (Markdown): Explains monitoring and error handling.
   - **Cell 16** (Code): **Implementation** – adds `monitored_query()` with timing, cost tracking, error handling, and retries; defines the `ask()` helper for quick queries.

**10. Testing & Results Display** (`Cells 17–18`):
   - **Cell 17** (Markdown): Quick reference for the three scores.
   - **Cell 18** (Code): **Implementation** – tests the advisor on sample questions, displays color‑coded responses, logs experiment metrics, and prints performance stats.

**11. Quantitative Evaluation** (`Cells 19–20`):
   - **Cell 19** (Markdown): Describes the evaluation methodology.
   - **Cell 20** (Code): **Implementation** – defines a `RAGEvaluator` class that generates RAG and LLM‑only answers, scores them with LLM‑as‑a‑judge, and compares performance.

**12. Results Analysis** (`Cells 21–22`):
   - **Cell 21** (Markdown): Explains how results are visualized and interpreted.
   - **Cell 22** (Code): **Implementation** – analyzes evaluation results, computes averages and improvement percentages, prints summary and stability metrics, and saves experiment data to CSV.

**13. Answer Comparison** (`Cells 23–24`):
   - **Cell 23** (Markdown): Introduces side‑by‑side comparison.
   - **Cell 24** (Code): **Implementation** – displays HTML side‑by‑side comparisons of RAG vs LLM answers, including score breakdowns.

**14. Interactive Interface** (`Cells 25–26`):
   - **Cell 25** (Markdown): Describes the interactive Q&A interface.
   - **Cell 26** (Code): **Implementation** – builds an ipywidgets‑based interface for submitting live queries and viewing answers with metrics.

---

### How the 4‑Tier Decision System Works

```
User Query
    ↓
Compute Relevance Score (with sigmoid transformation)
    ↓
    ┌──────────────────────────────┐
    │        Route by Score        │
    └──────────────────────────────┘
            │
    ┌───────┼───────┼───────┼
    ↓       ↓       ↓       ↓
 ≥0.65   0.50–0.65 0.15–0.50  <0.15
    ↓       ↓       ↓       ↓
  RAG    FALLBACK   LLM   OFF-TOPIC
  Mode   (Compare) DOMAIN  (Reject)
  🟢       🟢/🔵     CHECK      🔴
                    🟠/🔴
```

**Decision Logic Details:**
- **TIER 1 (RAG)**: Documents are highly relevant → use RAG directly.
- **TIER 2 (Fallback)**: Uncertain → run both RAG and LLM, score both, pick the winner.
- **TIER 3 (Domain Check)**: Low relevance → LLM checks if the query is stock‑related and evaluates confidence.
- **TIER 4 (Off‑Topic)**: Very low relevance → auto‑reject (save API cost).

---

### Key Innovations and Technical Concepts

**1. Finance‑Specific Embeddings**
- **Why**: General embeddings often fail to clearly separate finance vs non‑finance.
- **Solution**: Use a financial‑domain embedding model trained on sources like Bloomberg and Reuters.
- **Result**: Better separation between relevant and irrelevant queries.

**2. Sigmoid Transformation for Relevance Scores**
- **Problem**: Raw similarity scores are too close together (e.g., 0.45 vs 0.55).
- **Solution**: Apply a sigmoid function to amplify separation (e.g., 0.10 vs 0.85).
- **Formula**: `1 / (1 + exp(-steepness * (score - midpoint)))`.
- **Effect**: Creates sharper decision boundaries for more accurate routing.

**3. Smart Fallback with Answer Scoring**
- **Why**: It is often unclear whether documents or LLM knowledge will give the better answer.
- **Solution**: Generate both answers, score them on three metrics, and choose the winner.
- **Metrics**: Specificity, Relevance, Factuality (each 1–10, then averaged).

**4. LLM‑as‑a‑Judge Evaluation**
- **Why**: Need an objective, repeatable way to compare answer quality.
- **Solution**: Use the LLM itself as a consistent evaluator for both RAG and LLM‑only answers.
- **Benefit**: Enables reproducible, quantitative measurement of improvements.